# Provenance-governed agentic AI for personal knowledge and memory support
### Reference implementation · synthetic dataset · statistics · neural-network realisation

Executable companion to the manuscript *"Provenance-governed agentic AI for personal knowledge and memory support: a framework and rule-level demonstration"*.

**What this notebook does**

| § | Content | Paper |
|---|---|---|
| 1 | Generates a **synthetic** cohort of people with memory impairment (MCI, mild dementia, acquired brain injury) with full **metadata** (Frictionless Data Package) | §6.2 |
| 2 | Cohort statistics: ridgelines, cognitive landscape, metadata fingerprint | §6.2 |
| 3 | **Neural-network solution on real components** — system architecture and grounded-recall pipeline diagrams, bill of materials | §4, Fig. 1–2 |
| 4 | Paper scenarios (Olena, 72) on working code: Eqs. (1)–(5), invariants I1–I4 | §5 |
| 5 | Three conditions on identical histories — reminder app, single agent + RAG, proposed framework — with radar, Sankey, scoreboard, forest plot, animation | §6, Table 4 |
| 6 | Metadata × outcomes: raincloud, parallel coordinates, Kruskal–Wallis, Spearman, logistic regression | §6 |
| 7 | Recall by stratum: heatmap and sunburst (Table 3 counterexamples) | §5.2 |
| 8 | Release-rule calibration: risk–coverage curve, dev → held-out | §4.4 |
| 9 | **Live neural components**: mDeBERTa-XNLI entailment for ent(a, r), zero-shot intent, e5 dense retrieval, end-to-end neural recall | §4.4 |

> ⚠️ **All data are synthetic.** No record corresponds to a real person; distributions are design assumptions, not clinical estimates. Results show how the framework's rules behave and do **not** demonstrate benefit for people (paper §7.3).

*Runtime:* ≈2 min on CPU without §9; §9 downloads ≈1.3 GB of model weights — choose **Runtime → Change runtime type → T4 GPU** for speed (CPU also works).


## 0. Environment

In [ ]:
%pip -q install pandas numpy scipy matplotlib tabulate plotly
import os, sys
PROJECT = "pgmem_project"
os.makedirs(f"{PROJECT}/pgmem", exist_ok=True)
os.makedirs(f"{PROJECT}/data", exist_ok=True)
sys.path.insert(0, PROJECT)

### 0.1 Package code `pgmem`

The next cells write the package modules into `pgmem_project/pgmem/`. They can be collapsed.
*Alternative:* upload `pgmem_project.zip` and unzip it (cell after the modules).

In [ ]:
%%writefile pgmem_project/pgmem/__init__.py
"""pgmem — provenance-governed agentic memory support.

Reference implementation of the conceptual framework in
D. Prokopovych-Tkachenko, "Provenance-governed agentic AI for personal
knowledge and memory support: a conceptual framework".
"""
from .assistance import adapt_start_level, gate, next_level
from .governance import ActionQueue, PolicyEngine, QueuedAction
from .model import Episode, EpisodeState, EvidenceStatus, PersonalEventBase, Record
from .recall import Answer, Claim, GroundedRecall, NaiveRAG, Question, ReminderOnly, release, support_score

__version__ = "1.0.0"
__all__ = [
    "ActionQueue", "Answer", "Claim", "Episode", "EpisodeState", "EvidenceStatus", "GroundedRecall",
    "NaiveRAG", "PersonalEventBase", "PolicyEngine", "Question", "QueuedAction", "Record", "ReminderOnly",
    "adapt_start_level", "gate", "next_level", "release", "support_score",
]


In [ ]:
%%writefile pgmem_project/pgmem/model.py
"""Evidence-typed personal event base (Section 4.2, Equation 1).

Each record is r_i = (c_i, t_i, s_i, kappa_i, sigma_i, v_i, e_i):
content, time, source/provenance, confidence indicator, consent scope,
version and evidence status.  Corrections create new linked versions,
disagreements are marked as disputed, deletions leave only a minimal
audit event (no content).
"""
from __future__ import annotations

import itertools
from dataclasses import dataclass, field, replace
from datetime import datetime
from enum import Enum
from typing import Callable, Iterable


class EvidenceStatus(str, Enum):
    PLAN = "plan"                # entered as an intention / schedule
    SELF_REPORT = "self_report"  # the person said it happened
    OBSERVATION = "observation"  # sensor / external log observed it
    INFERENCE = "inference"      # drawn by the assistant
    DISPUTED = "disputed"        # incompatible records exist


# Which record statuses can support a claim that asserts a given status (I2).
# A plan supports "it was scheduled", never "it happened".
SUPPORTS: dict[EvidenceStatus, set[EvidenceStatus]] = {
    EvidenceStatus.PLAN: {EvidenceStatus.PLAN, EvidenceStatus.SELF_REPORT, EvidenceStatus.OBSERVATION},
    EvidenceStatus.SELF_REPORT: {EvidenceStatus.SELF_REPORT},
    EvidenceStatus.OBSERVATION: {EvidenceStatus.OBSERVATION},
    EvidenceStatus.INFERENCE: {EvidenceStatus.INFERENCE},
}

STATUS_STRENGTH = {
    EvidenceStatus.INFERENCE: 0,
    EvidenceStatus.PLAN: 1,
    EvidenceStatus.SELF_REPORT: 2,
    EvidenceStatus.OBSERVATION: 3,
}


@dataclass(frozen=True)
class Record:
    """One versioned entry of the personal event base."""

    record_id: str
    person_id: str
    subject: str                 # event identity, e.g. "appt:dr_kovalenko:2026-10-02"
    content: str                 # c_i
    time: datetime               # t_i
    source: str                  # s_i: calendar, user_voice, sensor:pillbox, carer_entry ...
    kappa: float                 # confidence indicator in [0, 1], not a probability
    scope: frozenset[str]        # sigma_i: recipients allowed to see it
    version: int = 1             # v_i
    status: EvidenceStatus = EvidenceStatus.PLAN  # e_i
    category: str = "general"    # data category used by consent scopes
    kind: str = "event"          # claim kind this record speaks to: scheduled/occurred/content
    value: str = ""              # normalised value of the claim (date, "true", note text)
    supersedes: str | None = None
    active: bool = True
    disputed: bool = False

    def __post_init__(self) -> None:
        if not 0.0 <= self.kappa <= 1.0:
            raise ValueError("kappa must lie in [0, 1]")


@dataclass
class AuditEvent:
    time: datetime
    actor: str
    action: str
    record_id: str
    detail: str = ""


class PersonalEventBase:
    """E = {r_1, ..., r_n} with versioning, disputes, deletion and audit."""

    def __init__(self, person_id: str):
        self.person_id = person_id
        self._records: dict[str, Record] = {}
        self.audit: list[AuditEvent] = []
        self._ids = itertools.count(1)
        self._listeners: list[Callable[[str], None]] = []

    # -- change notification (used by I4 cache invalidation) --------------
    def on_change(self, fn: Callable[[str], None]) -> None:
        self._listeners.append(fn)

    def _notify(self, subject: str) -> None:
        for fn in self._listeners:
            fn(subject)

    def new_id(self) -> str:
        return f"{self.person_id}-r{next(self._ids):05d}"

    # -- writes ------------------------------------------------------------
    def add(self, record: Record, actor: str = "system", when: datetime | None = None) -> Record:
        if record.person_id != self.person_id:
            raise ValueError("record belongs to another person")
        self._records[record.record_id] = record
        self.audit.append(AuditEvent(when or record.time, actor, "add", record.record_id, record.status.value))
        self._notify(record.subject)
        return record

    def correct(self, record_id: str, actor: str, when: datetime, **changes) -> Record:
        """Corrections never overwrite: a new linked version is created."""
        old = self._records[record_id]
        new = replace(old, record_id=self.new_id(), version=old.version + 1,
                      supersedes=old.record_id, active=True, **changes)
        self._records[old.record_id] = replace(old, active=False)
        self._records[new.record_id] = new
        self.audit.append(AuditEvent(when, actor, "correct", new.record_id, f"supersedes {old.record_id}"))
        self._notify(old.subject)
        return new

    def mark_disputed(self, record_ids: Iterable[str], actor: str, when: datetime) -> None:
        subjects = set()
        for rid in record_ids:
            r = self._records[rid]
            self._records[rid] = replace(r, disputed=True)
            subjects.add(r.subject)
            self.audit.append(AuditEvent(when, actor, "dispute", rid))
        for s in subjects:
            self._notify(s)

    def delete(self, record_id: str, actor: str, when: datetime) -> None:
        """Content is removed; the audit keeps only that a deletion happened."""
        r = self._records.pop(record_id)
        self.audit.append(AuditEvent(when, actor, "delete", record_id))
        self._notify(r.subject)

    # -- reads -------------------------------------------------------------
    def get(self, record_id: str) -> Record:
        return self._records[record_id]

    def all(self, include_inactive: bool = False) -> list[Record]:
        return [r for r in self._records.values() if include_inactive or r.active]

    def by_subject(self, subject: str, include_inactive: bool = False) -> list[Record]:
        return [r for r in self.all(include_inactive) if r.subject == subject]

    def __len__(self) -> int:
        return len(self._records)


class EpisodeState(str, Enum):
    """Explicit task-episode states (DP3, I3)."""

    PENDING = "pending"
    ACTIVE = "active"
    COMPLETED = "completed"   # always paired with a stated source
    POSTPONED = "postponed"
    DECLINED = "declined"
    UNKNOWN = "unknown"
    # Only produced by baselines that violate I3:
    MISSED_INFERRED = "missed_inferred"


@dataclass
class Episode:
    episode_id: str
    person_id: str
    task_type: str
    scheduled: datetime
    safety_critical: bool
    state: EpisodeState = EpisodeState.PENDING
    completion_source: str | None = None
    levels: list[int] = field(default_factory=list)

    def complete(self, source: str) -> None:
        if not source:
            raise ValueError("completion needs an explicit evidential basis (I3)")
        self.state = EpisodeState.COMPLETED
        self.completion_source = source


In [ ]:
%%writefile pgmem_project/pgmem/governance.py
"""Consent, access control and the dispatch-time permission check (I1, I4)."""
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime

from .model import AuditEvent


@dataclass
class PolicyEngine:
    """Revisable consent scopes keyed by (recipient, data category, action)."""

    person_id: str
    grants: set[tuple[str, str, str]] = field(default_factory=set)
    audit: list[AuditEvent] = field(default_factory=list)

    def grant(self, recipient: str, category: str, action: str) -> None:
        self.grants.add((recipient, category, action))

    def revoke(self, recipient: str, category: str, action: str) -> None:
        self.grants.discard((recipient, category, action))

    def allows(self, recipient: str, category: str, action: str) -> bool:
        # The person always sees and controls their own data.
        if recipient == self.person_id:
            return True
        return (recipient, category, action) in self.grants or (recipient, "*", action) in self.grants


@dataclass
class QueuedAction:
    key: str            # idempotency key: replay never duplicates (I4)
    kind: str           # message / disclosure / prompt
    recipient: str
    category: str
    payload: str
    created: datetime


class ActionQueue:
    """Actions are permitted at execution time, not at queueing time (I1)."""

    def __init__(self, policy: PolicyEngine):
        self.policy = policy
        self.pending: list[QueuedAction] = []
        self.sent: dict[str, QueuedAction] = {}
        self.cancelled: list[QueuedAction] = []

    def enqueue(self, action: QueuedAction) -> None:
        if action.key in self.sent or any(a.key == action.key for a in self.pending):
            return
        self.pending.append(action)

    def dispatch(self, now: datetime) -> list[QueuedAction]:
        delivered = []
        for a in self.pending:
            if a.key in self.sent:
                continue
            if self.policy.allows(a.recipient, a.category, a.kind):
                self.sent[a.key] = a
                delivered.append(a)
                self.policy.audit.append(AuditEvent(now, "assistant", f"send:{a.kind}", a.key, a.recipient))
            else:
                self.cancelled.append(a)
                self.policy.audit.append(AuditEvent(now, "assistant", "cancel", a.key, "permission changed"))
        self.pending = []
        return delivered


In [ ]:
%%writefile pgmem_project/pgmem/assistance.py
"""Graduated assistance (Section 4.3, Equations 2 and 3).

Levels: L0 silent context, L1 brief reminder, L2 contextual cue,
L3 step-by-step guidance, L4 trusted-person involvement.
"""
from __future__ import annotations

LEVEL_NAMES = {
    0: "L0 context, no prompt",
    1: "L1 brief reminder",
    2: "L2 contextual cue",
    3: "L3 step-by-step guidance",
    4: "L4 trusted-person involvement",
}


def gate(permission_ok: bool, postponed: bool, declined: bool, progress_supports_escalation: bool) -> int:
    """Binary gate G_k: 1 only if permission allows another prompt, the user has
    neither postponed nor declined, and the progress check supports escalation.
    Missing/ambiguous observations must yield progress_supports_escalation=False."""
    return int(permission_ok and not postponed and not declined and progress_supports_escalation)


def next_level(level: int, g: int, l_max: int) -> int:
    """Equation (2): L_{k+1} = G_k * min(L_k + 1, L_max) + (1 - G_k) * L_k."""
    return g * min(level + 1, l_max) + (1 - g) * level


def adapt_start_level(start: int, consecutive_without_escalation: int, m: int,
                      l_min: int, user_agrees: bool) -> int:
    """Equation (3): L0(j+1) = max(L0(j) - 1, L_min) after m consecutive
    episodes completed without escalation, and only with the user's agreement."""
    if consecutive_without_escalation >= m and user_agrees:
        return max(start - 1, l_min)
    return start


In [ ]:
%%writefile pgmem_project/pgmem/recall.py
"""Grounded recall and uncertainty (Section 4.4, Equations 4 and 5).

Three assistants share one interface ``answer(question) -> Answer``:

* ``GroundedRecall``  – the proposed policy (permission filter, evidence-type
  check, conflict/identity check before thresholding, release rule).
* ``NaiveRAG``        – a single-agent retrieval assistant used as a baseline.
* ``ReminderOnly``    – a reminder interface that cannot answer questions.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable

import numpy as np

from .governance import PolicyEngine
from .model import STATUS_STRENGTH, SUPPORTS, EvidenceStatus, PersonalEventBase, Record

THETA_LOW = 0.40
THETA_HIGH = 0.75


@dataclass(frozen=True)
class Question:
    question_id: str
    person_id: str
    requester: str            # who is asking (the person or a trusted person)
    subject: str              # event identity the question refers to
    kind: str                 # scheduled / occurred / content
    category: str
    text: str = ""


@dataclass(frozen=True)
class Claim:
    """An atomic claim a_j (Min et al., 2023)."""

    subject: str
    kind: str
    value: str
    asserted_status: EvidenceStatus
    record_id: str | None     # None => no record behind the claim (fabrication)
    record_version: int | None = None


@dataclass
class Answer:
    decision: str             # answer / answer_uncertain / abstain / clarify / refuse / none
    claims: list[Claim] = field(default_factory=list)
    s_min: float | None = None
    text: str = ""
    disclosed_records: list[str] = field(default_factory=list)
    candidate: Claim | None = None   # the scored draft claim, kept even when withheld


    @property
    def substantive(self) -> bool:
        return self.decision in ("answer", "answer_uncertain")


def support_score(claim: Claim, records: list[Record],
                  sup: Callable[[Claim, Record], float]) -> float:
    """Equation (4): S(a_j) = max_{r in R(q)} sup(a_j, r) * kappa_r.
    Heuristic, neither a probability nor independent verification."""
    if not records:
        return 0.0
    return max(sup(claim, r) * r.kappa for r in records)


def release(s_min: float, theta_low: float = THETA_LOW, theta_high: float = THETA_HIGH) -> str:
    """Equation (5)."""
    if s_min >= theta_high:
        return "answer"
    if s_min >= theta_low:
        return "answer_uncertain"
    return "abstain"


def exact_sup(claim: Claim, r: Record) -> float:
    """Reference entailment: identity, kind, value and evidence type must match."""
    if r.subject != claim.subject or r.kind != claim.kind or r.value != claim.value:
        return 0.0
    return 1.0 if r.status in SUPPORTS[claim.asserted_status] else 0.0


def noisy_sup(rng: np.random.Generator, sd: float = 0.12) -> Callable[[Claim, Record], float]:
    """An imperfect entailment estimator, as any real NLI model would be."""
    def f(claim: Claim, r: Record) -> float:
        base = exact_sup(claim, r)
        return float(np.clip(base + rng.normal(0, sd) - (0.05 if base else 0.0), 0, 1))
    return f


_TEMPLATES = {
    ("scheduled", EvidenceStatus.PLAN): "Your calendar lists: {value}.",
    ("occurred", EvidenceStatus.SELF_REPORT): "You recorded on {time} that this was done. I have not independently verified it.",
    ("occurred", EvidenceStatus.OBSERVATION): "This was recorded by the source '{source}' on {time}.",
    ("content", EvidenceStatus.PLAN): "The note says: '{value}'.",
    ("content", EvidenceStatus.SELF_REPORT): "You noted: '{value}'.",
}


class GroundedRecall:
    """The proposed recall policy (Figure 2b)."""

    def __init__(self, base: PersonalEventBase, policy: PolicyEngine,
                 sup: Callable[[Claim, Record], float] = exact_sup,
                 theta_low: float = THETA_LOW, theta_high: float = THETA_HIGH,
                 identity_confusion: float = 0.0, rng: np.random.Generator | None = None,
                 retriever: Callable[[Question, list[Record]], list[Record]] | None = None):
        self.base, self.policy, self.sup = base, policy, sup
        self.theta_low, self.theta_high = theta_low, theta_high
        self.identity_confusion = identity_confusion
        self.retriever = retriever
        self.rng = rng or np.random.default_rng(0)
        self._cache: dict[tuple, tuple[Answer, tuple]] = {}
        base.on_change(self._invalidate)

    # I4: corrections, disputes and deletions invalidate cached answers.
    def _invalidate(self, subject: str) -> None:
        for key in [k for k in self._cache if k[1] == subject]:
            del self._cache[key]

    def _retrieve(self, q: Question) -> list[Record]:
        recs = [r for r in self.base.all()
                if self.policy.allows(q.requester, r.category, "read")]       # I1
        if self.retriever is not None:
            return self.retriever(q, recs)          # e.g. a dense neural retriever (pgmem.neural)
        hits = [r for r in recs if r.subject == q.subject]
        if self.identity_confusion and self.rng.random() < self.identity_confusion:
            # Retriever slips in a record about a similar but different event.
            similar = [r for r in recs if r.subject != q.subject and r.subject.split(":")[0] == q.subject.split(":")[0]]
            if similar:
                hits = hits + [similar[int(self.rng.integers(len(similar)))]]
        return hits

    def answer(self, q: Question) -> Answer:
        # Permissions travel with cached answers: any grant change is a cache miss (I1, I4).
        key = (q.requester, q.subject, q.kind, frozenset(self.policy.grants))
        if key in self._cache:
            return self._cache[key][0]
        ans = self._answer(q)
        self._cache[key] = (ans, tuple(c.record_id for c in ans.claims))
        return ans

    def _answer(self, q: Question) -> Answer:
        if not self.policy.allows(q.requester, q.category, "read"):
            return Answer("refuse", text="I cannot share this information without your permission.")
        hits = self._retrieve(q)
        if not hits:
            return Answer("abstain", text="I don't have any record of this, so I cannot know it.")
        # Conflict / identity checks precede thresholding: a max cannot resolve contradictions.
        if any(r.disputed for r in hits):
            return Answer("clarify", text="The records about this contradict each other. Let's clarify it together.")
        if len({r.subject for r in hits}) > 1:
            return Answer("clarify", text="I found records about several similar events. Which one do you mean?")
        values = {(r.kind, r.value) for r in hits if r.kind == q.kind}
        if len(values) > 1:
            return Answer("clarify", text="There are several incompatible records. Let's clarify.")

        # Draft: the strongest claim the records allow, never stronger (I2).
        same_kind = [r for r in hits if r.kind == q.kind]
        limit_note = ""
        if q.kind == "occurred" and not same_kind:
            # Only a plan exists: answer about the schedule and state the limit.
            same_kind = [r for r in hits if r.kind == "scheduled"]
            limit_note = " These records do not confirm that the event took place."
        if not same_kind:
            return Answer("abstain", text="The records do not address this exact question.")
        best = max(same_kind, key=lambda r: (STATUS_STRENGTH.get(r.status, 0), r.time))
        claim = Claim(best.subject, best.kind, best.value, best.status, best.record_id, best.version)
        s = support_score(claim, hits, self.sup)
        decision = release(s, self.theta_low, self.theta_high)
        if decision == "abstain":
            return Answer("abstain", s_min=s, text="I am not sure and I will not invent an answer.", candidate=claim)
        tpl = _TEMPLATES.get((claim.kind, claim.asserted_status), "Record: {value}.")
        text = tpl.format(value=best.value, source=best.source, time=best.time.strftime("%d %b at %H:%M")) + limit_note
        if decision == "answer_uncertain":
            text += " (Record support is incomplete — worth checking.)"
        return Answer(decision, [claim], s, text, [best.record_id], candidate=claim)


class NaiveRAG:
    """Single-agent retrieval assistant without evidence typing or governance."""

    def __init__(self, base: PersonalEventBase, rng: np.random.Generator,
                 p_fabricate: float = 0.4, p_stale: float = 0.3):
        self.base, self.rng = base, rng
        self.p_fabricate, self.p_stale = p_fabricate, p_stale

    def answer(self, q: Question) -> Answer:
        pool = self.base.all(include_inactive=self.rng.random() < self.p_stale)
        hits = [r for r in pool if r.subject == q.subject]          # no permission filter
        if not hits:
            if self.rng.random() < self.p_fabricate:
                fake = Claim(q.subject, q.kind, "true" if q.kind == "occurred" else "invented",
                             EvidenceStatus.OBSERVATION, None)
                return Answer("answer", [fake], 1.0, "Yes, that happened.")
            return Answer("abstain", text="I don't know.")
        best = max(hits, key=lambda r: r.kappa)                       # highest score wins
        value = best.value
        if q.kind == "occurred" and best.kind == "scheduled":
            value = "true"                                            # plan read as attendance
        claim = Claim(q.subject, q.kind, value, EvidenceStatus.OBSERVATION, best.record_id, best.version)
        return Answer("answer", [claim], best.kappa, f"Yes: {value}.", [best.record_id])


class ReminderOnly:
    def answer(self, q: Question) -> Answer:
        return Answer("none", text="(the reminder interface does not answer questions)")


In [ ]:
%%writefile pgmem_project/pgmem/metrics.py
"""Evaluation indicators (Section 6, Equations 6 and 7)."""
from __future__ import annotations

import math


def _ratio(num: float, den: float) -> float:
    return num / den if den else math.nan


def tsr(n_completed: int, n_scheduled: int) -> float:
    """Task success rate TSR = N_c / N_s."""
    return _ratio(n_completed, n_scheduled)


def fpr(n_false_prompts: int, n_prompts: int) -> float:
    """False-prompt rate FPR = N_f / N_p."""
    return _ratio(n_false_prompts, n_prompts)


def f_ext(n_supported: int, n_released: int) -> float:
    """Externally assessed recall faithfulness F_ext = N_e / N_r.
    Undefined (NaN) when no claim is released — never reported as perfect."""
    return _ratio(n_supported, n_released)


def coverage(n_substantive: int, n_questions: int) -> float:
    """Answer coverage C = N_a / N_q."""
    return _ratio(n_substantive, n_questions)


def independence_index(n_completed_l0_l1: int, n_completed: int) -> float:
    """Share of completions achieved at L0-L1."""
    return _ratio(n_completed_l0_l1, n_completed)


def per_100(n_events: int, n_answers: int) -> float:
    """E.g. fabricated recollections per 100 substantive answers."""
    return 100.0 * _ratio(n_events, n_answers)


In [ ]:
%%writefile pgmem_project/pgmem/synth.py
"""Synthetic cohort and personal histories (Section 6, stage 1).

IMPORTANT: every value produced here is synthetic.  Distributions are
plausible design assumptions for people with memory impairment (MCI,
mild dementia, acquired brain injury), not empirical estimates, and no
generated person corresponds to a real individual.
"""
from __future__ import annotations

from datetime import datetime, timedelta

import numpy as np
import pandas as pd

START = datetime(2026, 9, 1)

DIAGNOSES = {
    # name: (share, age mean, age sd, age min, age max, MoCA mean, MoCA sd, CDR global)
    "MCI": (0.50, 73, 6.5, 55, 92, 22.5, 2.0, 0.5),
    "mild_dementia": (0.25, 78, 6.0, 60, 95, 17.5, 2.5, 1.0),
    "ABI": (0.25, 46, 14.0, 18, 80, 23.5, 3.0, np.nan),
}

TASKS = {
    # task: (every n days, safety critical, sensorable, hour, category)
    "medication_morning": (1, True, True, 8, "medication"),
    "daily_walk": (1, False, True, 11, "activity"),
    "appointment_prep": (7, False, False, 9, "appointments"),
    "bill_payment": (14, False, False, 15, "finance"),
    "call_family": (3, False, False, 18, "social"),
}

DOCTORS = ["Kovalenko", "Shevchuk", "Bondar", "Melnyk", "Tkachenko", "Kravets"]
RELATIVES = ["Andrii", "Iryna", "Oleh", "Mariia", "Taras", "Nataliia"]
NOTES = ["next check-up in three months", "buy bread and milk", "call the housing office",
         "Wi-Fi password is on the fridge", "collect glasses from repair", "wish the neighbour a happy birthday"]


def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def generate_cohort(n: int = 240, seed: int = 2026) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    names = list(DIAGNOSES)
    shares = np.array([DIAGNOSES[d][0] for d in names])
    dx = rng.choice(names, size=n, p=shares / shares.sum())
    rows = []
    for i, d in enumerate(dx):
        _, am, asd, amin, amax, mm, msd, cdr = DIAGNOSES[d]
        age = int(np.clip(round(rng.normal(am, asd)), amin, amax))
        moca = int(np.clip(round(rng.normal(mm, msd)), 8, 30))
        edu = int(np.clip(round(rng.normal(13, 3)), 6, 22))
        # MoCA adds one point for <= 12 years of education
        moca_adj = min(30, moca + (1 if edu <= 12 else 0))
        ability = (moca - 22) / 4 + rng.normal(0, 0.5)
        # PRMQ subscales: 8-40, higher = more frequent everyday failures
        prmq_p = int(np.clip(round(24 - 4.5 * ability + rng.normal(0, 3)), 8, 40))
        prmq_r = int(np.clip(round(22 - 4.0 * ability + rng.normal(0, 3)), 8, 40))
        older = age >= 65
        lives_alone = bool(rng.random() < (0.45 if older else 0.30))
        trusted = bool(rng.random() < 0.85)
        literacy = int(np.clip(round(rng.normal(3.4 - (0.6 if older else 0) + 0.3 * ability, 0.9)), 1, 5))
        rows.append(dict(
            participant_id=f"P{i + 1:03d}",
            age=age,
            age_band=pd.cut([age], [0, 49, 64, 74, 84, 120], labels=["<50", "50-64", "65-74", "75-84", "85+"])[0],
            sex=rng.choice(["F", "M"], p=[0.58, 0.42] if older else [0.40, 0.60]),
            diagnosis=d,
            cdr_global=cdr,
            moca=moca_adj,
            education_years=edu,
            prmq_prospective=prmq_p,
            prmq_retrospective=prmq_r,
            months_since_diagnosis=int(rng.gamma(2.0, 14.0)),
            lives_alone=lives_alone,
            has_trusted_person=trusted,
            trusted_relation=(rng.choice(["daughter", "son", "spouse", "sibling", "friend"]) if trusted else "none"),
            device=rng.choice(["smartphone", "tablet", "both"], p=[0.5, 0.2, 0.3]),
            digital_literacy=literacy,
            uses_voice=bool(rng.random() < 0.55),
            sensors_connected=bool(rng.random() < 0.6),
            call_log_connected=bool(rng.random() < 0.5),
            language=rng.choice(["uk", "en"], p=[0.8, 0.2]),
            agrees_to_level_reduction=bool(rng.random() < 0.7),
            ability_latent=round(float(ability), 3),
        ))
    return pd.DataFrame(rows)


def generate_episodes(cohort: pd.DataFrame, days: int = 28) -> pd.DataFrame:
    rows = []
    for p in cohort.itertuples():
        offset = int(p.participant_id[1:]) % 7
        for day in range(days):
            for task, (every, safety, sensorable, hour, cat) in TASKS.items():
                if (day + offset) % every:
                    continue
                rows.append(dict(
                    episode_id=f"{p.participant_id}-E{len(rows):06d}",
                    participant_id=p.participant_id,
                    day=day,
                    task_type=task,
                    category=cat,
                    safety_critical=safety,
                    sensorable=sensorable and p.sensors_connected,
                    scheduled=(START + timedelta(days=day, hours=hour)).isoformat(),
                ))
    return pd.DataFrame(rows)


def _rec(rows, pid, subject, content, t, source, kappa, scope, status, category, kind, value,
         version=1, supersedes="", active=True, disputed=False):
    rid = f"{pid}-r{sum(1 for r in rows if r['person_id'] == pid) + 1:05d}"
    rows.append(dict(record_id=rid, person_id=pid, subject=subject, content=content, time=t.isoformat(),
                     source=source, kappa=round(kappa, 3), scope=scope, version=version, status=status,
                     category=category, kind=kind, value=value, supersedes=supersedes,
                     active=active, disputed=disputed))
    return rid


def generate_memory_corpus(cohort: pd.DataFrame, seed: int = 7):
    """Personal event base records, recall questions and the independent
    reference history used to judge released claims (Section 6)."""
    rng = np.random.default_rng(seed)
    records, questions, reference = [], [], []

    def ref(pid, subject, kind, value):
        reference.append(dict(person_id=pid, subject=subject, kind=kind, value=value))

    def ask(pid, stratum, requester, subject, kind, category, ideal, text):
        questions.append(dict(question_id=f"{pid}-Q{sum(1 for q in questions if q['person_id'] == pid) + 1:03d}",
                              person_id=pid, stratum=stratum, requester=requester, subject=subject,
                              kind=kind, category=category, ideal=ideal, text=text))

    for p in cohort.itertuples():
        pid = p.participant_id
        trusted = f"trusted:{pid}"
        p_report = float(_sigmoid(0.3 + 0.8 * p.ability_latent))

        # 1-2. Past appointments: a plan; attendance only if self-reported.
        for j in range(3):
            doc = DOCTORS[(j + int(pid[1:])) % len(DOCTORS)]
            d = START + timedelta(days=int(rng.integers(0, 28)), hours=10)
            subj = f"appt:{doc}:{d.date()}"
            _rec(records, pid, subj, f"Appointment with Dr {doc}", d, "calendar", 0.95,
                 "self", "plan", "appointments", "scheduled", str(d.date()))
            ref(pid, subj, "scheduled", str(d.date()))
            attended = rng.random() < 0.9
            ref(pid, subj, "occurred", str(attended).lower())
            reported = attended and rng.random() < p_report
            if reported:
                _rec(records, pid, subj, "Attended the appointment", d + timedelta(hours=3), "user_voice", 0.8,
                     "self", "self_report", "appointments", "occurred", "true")
            ref(pid, subj, "reported", "true" if reported else "none")
            ask(pid, "answerable_plan", pid, subj, "scheduled", "appointments", "answer",
                f"When was my appointment with Dr {doc} scheduled?")
            ask(pid, "attendance", pid, subj, "occurred", "appointments", "answer",
                f"Did I attend my appointment with Dr {doc}?")

        # 3. Medication: self-reports (which can themselves be wrong) or nothing.
        for j in range(4):
            d = START + timedelta(days=int(rng.integers(0, 28)), hours=8)
            subj = f"med:morning:{d.date()}"
            taken = rng.random() < 0.85
            reported = rng.random() < (p_report if taken else 0.05)
            ref(pid, subj, "occurred", str(taken).lower())
            ref(pid, subj, "reported", "true" if reported else "none")
            if reported:
                _rec(records, pid, subj, "Morning medication taken", d + timedelta(minutes=47), "user_touch",
                     0.85, "self,trusted", "self_report", "medication", "occurred", "true")
            ask(pid, "self_report" if reported else "unanswerable", pid, subj, "occurred", "medication",
                "answer" if reported else "abstain", f"Did I take my morning medication on {d.date()}?")

        # 4. Calls: answerable only when the call log is connected.
        for j in range(2):
            who = RELATIVES[(j + int(pid[1:])) % len(RELATIVES)]
            d = START + timedelta(days=int(rng.integers(0, 28)))
            subj = f"call:{who}:{d.date()}"
            called = rng.random() < 0.5
            ref(pid, subj, "occurred", str(called).lower())
            if p.call_log_connected and called:
                _rec(records, pid, subj, f"Incoming call: {who}", d + timedelta(hours=19), "call_log",
                     0.98, "self", "observation", "social", "occurred", "true")
                stratum, ideal = "answerable_observation", "answer"
            else:
                stratum, ideal = "unanswerable", "abstain"
            ask(pid, stratum, pid, subj, "occurred", "social", ideal, f"Did {who} call me on {d.date()}?")

        # 5. Contradiction: a self-report and a carer entry disagree.
        d = START + timedelta(days=int(rng.integers(0, 28)), hours=15)
        subj = f"bill:electricity:{d.date()}"
        ref(pid, subj, "occurred", str(rng.random() < 0.6).lower())
        _rec(records, pid, subj, "Electricity bill paid", d, "user_voice", 0.8,
             "self", "self_report", "finance", "occurred", "true", disputed=True)
        _rec(records, pid, subj, "Bill not paid (carer entry)", d + timedelta(hours=5),
             "carer_entry", 0.85, "self", "observation", "finance", "occurred", "false", disputed=True)
        ask(pid, "contradictory", pid, subj, "occurred", "finance", "clarify",
            "Did I pay the electricity bill?")

        # 6. Stale: a rescheduled appointment (v1 superseded by v2).
        d1 = START + timedelta(days=int(rng.integers(0, 20)), hours=12)
        d2 = d1 + timedelta(days=int(rng.integers(2, 8)))
        subj = f"appt:dentist:{pid}"
        old = _rec(records, pid, subj, "Dentist", d1, "calendar", 0.95, "self", "plan",
                   "appointments", "scheduled", str(d1.date()), active=False)
        _rec(records, pid, subj, "Dentist (rescheduled)", d2, "calendar", 0.95, "self", "plan",
             "appointments", "scheduled", str(d2.date()), version=2, supersedes=old)
        ref(pid, subj, "scheduled", str(d2.date()))
        ask(pid, "stale", pid, subj, "scheduled", "appointments", "answer", "When is my dentist appointment?")

        # 7-8. Personal notes: the person may read them, a trusted person may not.
        for j in range(2):
            note = NOTES[(j * 3 + int(pid[1:])) % len(NOTES)]
            d = START + timedelta(days=int(rng.integers(0, 28)), hours=20)
            subj = f"note:{pid}:{j}"
            _rec(records, pid, subj, note, d, "user_voice", 0.9, "self", "self_report",
                 "personal_note", "content", note)
            ref(pid, subj, "content", note)
            ref(pid, subj, "reported", note)
            ask(pid, "answerable_note", pid, subj, "content", "personal_note", "answer", "What did I write in my note?")
            ask(pid, "consent_restricted", trusted, subj, "content", "personal_note", "refuse",
                "(trusted person) What is written in her personal note?")

    return pd.DataFrame(records), pd.DataFrame(questions), pd.DataFrame(reference)


In [ ]:
%%writefile pgmem_project/pgmem/simulate.py
"""Comparative simulation on identical synthetic histories (Section 6).

Conditions: ``reminder`` (fixed reminder interface), ``single_agent``
(retrieval assistant that escalates on timeouts and infers outcomes from
missing data) and ``proposed`` (the action-and-evidence contract).
Person behaviour uses common random numbers, so the three conditions
face exactly the same simulated people and days.
"""
from __future__ import annotations

from datetime import datetime

import numpy as np
import pandas as pd

from .assistance import adapt_start_level, gate, next_level
from .governance import PolicyEngine
from .model import STATUS_STRENGTH, SUPPORTS, EvidenceStatus, PersonalEventBase, Record
from .recall import GroundedRecall, NaiveRAG, Question, ReminderOnly, noisy_sup

CONDITIONS = ["reminder", "single_agent", "proposed"]
MAX_PROMPTS = 4
P_POSTPONE, P_DECLINE, P_DEVICE_DOWN, P_WITHDRAWN = 0.07, 0.03, 0.04, 0.08
M_ADAPT, L_MIN = 5, 0


def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def _respond(u: float, level: int, a: float) -> str:
    p_c = 0.8 if level >= 4 else float(_sigmoid(-1.4 + 0.7 * level + 0.9 * a))
    if u < p_c:
        return "complete"
    u -= p_c
    if u < P_POSTPONE:
        return "postpone"
    u -= P_POSTPONE
    if u < P_DECLINE:
        return "decline"
    rest = 1 - p_c - P_POSTPONE - P_DECLINE
    return "needs_help" if u < rest / 2 else "no_response"


def simulate_episodes(cohort: pd.DataFrame, episodes: pd.DataFrame, seed: int = 11) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n = len(episodes)
    draws = dict(
        before=rng.random(n), known_before=rng.random(n), sr=rng.random(n), sensor=rng.random(n),
        withdrawn=rng.random(n), resp=rng.random((n, MAX_PROMPTS + 1)), dev=rng.random((n, MAX_PROMPTS + 1)),
    )
    people = cohort.set_index("participant_id")
    out = []
    for cond in CONDITIONS:
        start_level: dict[tuple[str, str], int] = {}
        streak: dict[tuple[str, str], int] = {}
        for i, ep in enumerate(episodes.itertuples()):
            p = people.loc[ep.participant_id]
            a = p.ability_latent
            p_sr = float(_sigmoid(0.3 + 0.8 * a))
            detected = bool(ep.sensorable and draws["sensor"][i] < 0.85)
            reports = draws["sr"][i] < p_sr
            withdrawn = draws["withdrawn"][i] < P_WITHDRAWN
            l_max = 4 if p.has_trusted_person else 3
            key = (ep.participant_id, ep.task_type)
            s0 = start_level.get(key, 1) if cond == "proposed" else 1

            done = draws["before"][i] < float(_sigmoid(-0.6 + 0.9 * a))
            completion_level = 0 if done else None
            known = None
            if done and detected and cond != "reminder":
                known = "sensor"
            elif done and draws["known_before"][i] < 0.4 * p_sr:
                known = "self_report"

            prompts = false_prompts = missed = escalations = 0
            unauthorised = cancelled = False
            level, max_level = s0, s0
            postponed = declined = resumed = ignored_decline = False
            k = 0
            while known is None and k < MAX_PROMPTS and not declined:
                if postponed:
                    if cond == "proposed":
                        if resumed:
                            break
                        resumed = True       # the user's chosen resumption rule: one later prompt
                        postponed = False
                    elif cond == "reminder":
                        postponed = False    # snooze -> next reminder is the resumption
                if level == 0:
                    # L0: context retained, no prompt; progress check before L1.
                    level = 1
                    max_level = max(max_level, 1)
                    continue
                if level == 4:
                    if cond == "proposed" and withdrawn:
                        cancelled = True     # I1: recheck at dispatch -> cancel
                        level = 3
                        l_max = 3
                        continue
                    if withdrawn:
                        unauthorised = True
                if draws["dev"][i, k] < P_DEVICE_DOWN:
                    missed += 1
                    k += 1
                    continue
                prompts += 1
                max_level = max(max_level, level)
                if done or postponed or ignored_decline:
                    false_prompts += 1
                k += 1
                if done:
                    if draws["resp"][i, k - 1] < p_sr:
                        known = "self_report"   # "Have you already...?" -> "yes"
                    continue
                r = _respond(draws["resp"][i, k - 1], level, a)
                if r == "complete":
                    done, completion_level = True, level
                    if reports:
                        known = "self_report"
                    elif detected and cond != "reminder":
                        known = "sensor"
                    continue
                if r == "postpone":
                    postponed = True
                    if cond == "single_agent":   # treats the delay as failure and escalates
                        new = next_level(level, 1, l_max)
                        escalations += new > level
                        level = new
                    continue
                if r == "decline":
                    declined = cond != "reminder"
                    ignored_decline = cond == "reminder"  # the reminder app keeps reminding
                    continue
                if cond == "reminder":
                    continue                  # fixed L1, no escalation
                if cond == "single_agent":
                    g = 1                     # timeout alone triggers escalation
                else:
                    g = gate(permission_ok=True, postponed=postponed, declined=declined,
                             progress_supports_escalation=(r == "needs_help"))
                new = next_level(level, g, l_max)
                escalations += new > level
                level = new

            # Recorded state of the episode
            if known:
                state, source = "completed", known
            elif declined:
                state, source = "declined", "user"
            elif postponed and cond == "proposed":
                state, source = "postponed", "user"
            elif cond == "proposed":
                state, source = "unknown", ""
            else:
                state, source = "missed_inferred", ""  # silence converted to failure (violates I3)

            if cond == "proposed":
                if state == "completed" and escalations == 0:
                    streak[key] = streak.get(key, 0) + 1
                else:
                    streak[key] = 0
                if not ep.safety_critical:
                    new_start = adapt_start_level(s0, streak[key], M_ADAPT, L_MIN, bool(p.agrees_to_level_reduction))
                    if new_start != s0:
                        streak[key] = 0
                    start_level[key] = new_start

            out.append(dict(
                condition=cond, episode_id=ep.episode_id, participant_id=ep.participant_id,
                task_type=ep.task_type, safety_critical=ep.safety_critical, day=ep.day,
                start_level=s0, max_level=max_level, prompts=prompts, false_prompts=false_prompts,
                missed_prompts=missed, escalations=escalations, completed_true=done,
                completion_level=completion_level, recorded_state=state, evidence_source=source,
                invented_outcome=state == "missed_inferred",
                wrong_failure=(state == "missed_inferred") and done,
                unauthorised_contact=unauthorised, contact_cancelled=cancelled,
            ))
    return pd.DataFrame(out)


# --------------------------------------------------------------------- recall
def _to_record(row) -> Record:
    return Record(
        record_id=row.record_id, person_id=row.person_id, subject=row.subject, content=row.content,
        time=datetime.fromisoformat(row.time), source=row.source, kappa=float(row.kappa),
        scope=frozenset(str(row.scope).split(",")), version=int(row.version),
        status=EvidenceStatus(row.status), category=row.category, kind=row.kind, value=str(row.value),
        supersedes=row.supersedes or None, active=bool(row.active), disputed=bool(row.disputed),
    )


def build_event_base(person_id: str, records: pd.DataFrame) -> PersonalEventBase:
    base = PersonalEventBase(person_id)
    for row in records[records.person_id == person_id].fillna({"supersedes": ""}).itertuples():
        base.add(_to_record(row), actor=row.source)
    return base


def build_policy(person_id: str) -> PolicyEngine:
    pol = PolicyEngine(person_id)
    trusted = f"trusted:{person_id}"
    pol.grant(trusted, "medication", "read")      # e.g. missed safety-critical tasks
    pol.grant(trusted, "medication", "message")
    return pol


def _ext_supported(claim, ref: dict) -> bool:
    if claim.kind == "occurred" and claim.asserted_status == EvidenceStatus.SELF_REPORT:
        return ref.get((claim.subject, "reported")) == claim.value
    return ref.get((claim.subject, claim.kind)) == claim.value


def simulate_recall(cohort, records, questions, reference, seed: int = 13, sup_factory=None) -> pd.DataFrame:
    """``sup_factory(rng) -> sup(claim, record)`` swaps the entailment estimator
    of the proposed condition, e.g. ``lambda rng: neural.neural_sup(scorer)``."""
    rng = np.random.default_rng(seed)
    sup_factory = sup_factory or noisy_sup
    out = []
    for pid in cohort.participant_id:
        recs = records[records.person_id == pid]
        ref = {(r.subject, r.kind): str(r.value) for r in reference[reference.person_id == pid].itertuples()}
        base = build_event_base(pid, recs)
        policy = build_policy(pid)
        by_id = {r.record_id: r for r in base.all(include_inactive=True)}
        assistants = {
            "reminder": ReminderOnly(),
            "single_agent": NaiveRAG(base, np.random.default_rng(rng.integers(1 << 31))),
            "proposed": GroundedRecall(base, policy, sup=sup_factory(np.random.default_rng(rng.integers(1 << 31))),
                                       identity_confusion=0.03, rng=np.random.default_rng(rng.integers(1 << 31))),
        }
        for q in questions[questions.person_id == pid].itertuples():
            question = Question(q.question_id, pid, q.requester, q.subject, q.kind, q.category, q.text)
            for cond, asst in assistants.items():
                ans = asst.answer(question)
                n_sup = n_fab = n_over = n_stale = n_wrong = 0
                for c in ans.claims:
                    if c.subject != q.subject:
                        n_wrong += 1          # accurate record, wrong question: unsupported
                    else:
                        n_sup += _ext_supported(c, ref)
                    if c.record_id is None:
                        n_fab += 1
                        continue
                    r = by_id[c.record_id]
                    n_over += r.status not in SUPPORTS.get(c.asserted_status, set()) or \
                        STATUS_STRENGTH.get(c.asserted_status, 0) > STATUS_STRENGTH.get(r.status, 0)
                    n_stale += not r.active
                cand = ans.candidate
                cand_supported = bool(cand is not None and cand.subject == q.subject and _ext_supported(cand, ref))
                disclosed = any(not policy.allows(q.requester, by_id[rid].category, "read")
                                for rid in ans.disclosed_records)
                decision_class = "answer" if ans.substantive else ans.decision
                out.append(dict(
                    condition=cond, question_id=q.question_id, participant_id=pid, stratum=q.stratum,
                    ideal=q.ideal, decision=ans.decision, substantive=ans.substantive,
                    appropriate=decision_class == q.ideal or (q.ideal == "refuse" and ans.decision == "abstain"),
                    claims_released=len(ans.claims), claims_supported=n_sup, fabricated=n_fab,
                    overstated=n_over, stale_used=n_stale, wrong_event=n_wrong, consent_violation=disclosed,
                    s_min=ans.s_min, cand_supported=cand_supported, answer_text=ans.text,
                ))
    return pd.DataFrame(out)


In [ ]:
%%writefile pgmem_project/pgmem/stats.py
"""Statistics over the cohort metadata and the simulated outcomes."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

from . import metrics as M


def describe_cohort(cohort: pd.DataFrame) -> pd.DataFrame:
    """Table of cohort characteristics by diagnosis (mean ± SD, %)."""
    def summarise(g: pd.DataFrame) -> pd.Series:
        ms = lambda c: f"{g[c].mean():.1f} ± {g[c].std():.1f}"
        pct = lambda c: f"{100 * g[c].mean():.0f}%"
        return pd.Series({
            "N": len(g),
            "Age, years": ms("age"),
            "Women": f"{100 * (g.sex == 'F').mean():.0f}%",
            "MoCA (0–30)": ms("moca"),
            "PRMQ prospective (8–40)": ms("prmq_prospective"),
            "PRMQ retrospective (8–40)": ms("prmq_retrospective"),
            "Education, years": ms("education_years"),
            "Months since diagnosis": ms("months_since_diagnosis"),
            "Lives alone": pct("lives_alone"),
            "Has trusted person": pct("has_trusted_person"),
            "Digital literacy (1–5)": ms("digital_literacy"),
            "Sensors connected": pct("sensors_connected"),
            "Call log connected": pct("call_log_connected"),
        })
    table = cohort.groupby("diagnosis").apply(summarise, include_groups=False).T
    table["Total"] = summarise(cohort)
    return table


def memory_impairment_profile(cohort: pd.DataFrame) -> pd.DataFrame:
    """Severity bands from MoCA (common cut-offs: >=26 normal, 18–25 mild,
    10–17 moderate, <10 severe) crossed with diagnosis."""
    bands = pd.cut(cohort.moca, [-1, 9, 17, 25, 30], labels=["<10 severe", "10–17 moderate", "18–25 mild", "26–30 normal"])
    return pd.crosstab(cohort.diagnosis, bands, margins=True, margins_name="Total")


def participant_metrics(ep: pd.DataFrame, rc: pd.DataFrame) -> pd.DataFrame:
    """Per participant x condition indicators (Eq. 6–7 and Section 6 extras)."""
    e = ep.assign(l01=ep.completed_true & (ep.completion_level.fillna(9) <= 1))
    eg = e.groupby(["participant_id", "condition"]).agg(
        n_scheduled=("episode_id", "size"), n_completed=("completed_true", "sum"),
        n_prompts=("prompts", "sum"), n_false=("false_prompts", "sum"), n_l01=("l01", "sum"),
        n_missed=("missed_prompts", "sum"), n_invented=("invented_outcome", "sum"),
        n_wrong_failure=("wrong_failure", "sum"), n_unauth=("unauthorised_contact", "sum"),
    )
    r = rc.assign(
        unnecessary_abstain=rc.decision.isin(["abstain", "refuse"]) & (rc.ideal == "answer"),
        appropriate_abstain=rc.decision.isin(["abstain", "refuse"]) & rc.ideal.isin(["abstain", "refuse"]),
        should_abstain=rc.ideal.isin(["abstain", "refuse"]),
    )
    rg = r.groupby(["participant_id", "condition"]).agg(
        n_questions=("question_id", "size"), n_substantive=("substantive", "sum"),
        n_released=("claims_released", "sum"), n_supported=("claims_supported", "sum"),
        n_fabricated=("fabricated", "sum"), n_overstated=("overstated", "sum"),
        n_stale=("stale_used", "sum"), n_consent=("consent_violation", "sum"),
        n_unnec_abstain=("unnecessary_abstain", "sum"), n_appr_abstain=("appropriate_abstain", "sum"),
        n_should_abstain=("should_abstain", "sum"), n_appropriate=("appropriate", "sum"),
    )
    pm = eg.join(rg).reset_index()
    return add_rates(pm)


def add_rates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    v = np.vectorize
    df["TSR"] = v(M.tsr)(df.n_completed, df.n_scheduled)
    df["FPR"] = v(M.fpr)(df.n_false, df.n_prompts)
    df["independence"] = v(M.independence_index)(df.n_l01, df.n_completed)
    df["invented_outcome_rate"] = df.n_invented / df.n_scheduled
    df["F_ext"] = v(M.f_ext)(df.n_supported, df.n_released)
    df["coverage"] = v(M.coverage)(df.n_substantive, df.n_questions)
    df["fabricated_per_100"] = v(M.per_100)(df.n_fabricated, df.n_substantive)
    df["overstated_per_100"] = v(M.per_100)(df.n_overstated, df.n_substantive)
    df["appropriate_abstention"] = v(M._ratio)(df.n_appr_abstain, df.n_should_abstain)
    df["appropriate_behaviour"] = df.n_appropriate / df.n_questions
    return df


RATE_COLS = ["TSR", "FPR", "independence", "invented_outcome_rate", "F_ext", "coverage",
             "fabricated_per_100", "overstated_per_100", "appropriate_abstention", "appropriate_behaviour"]
COUNT_COLS = ["n_scheduled", "n_completed", "n_prompts", "n_false", "n_l01", "n_missed", "n_invented",
              "n_wrong_failure", "n_unauth", "n_questions", "n_substantive", "n_released", "n_supported",
              "n_fabricated", "n_overstated", "n_stale", "n_consent", "n_unnec_abstain", "n_appr_abstain",
              "n_should_abstain", "n_appropriate"]


def pooled(pm: pd.DataFrame, by: list[str]) -> pd.DataFrame:
    """Ratio-of-sums indicators for any grouping."""
    return add_rates(pm.groupby(by)[COUNT_COLS].sum()).reset_index()


def cluster_bootstrap(pm: pd.DataFrame, metric: str, n_boot: int = 1000, seed: int = 0) -> pd.DataFrame:
    """Pooled metric per condition with 95% CIs resampling participants (clusters)."""
    rng = np.random.default_rng(seed)
    pids = pm.participant_id.unique()
    idx = {p: g for p, g in pm.groupby("participant_id")}
    point = pooled(pm, ["condition"]).set_index("condition")[metric]
    boots = []
    for _ in range(n_boot):
        sample = pd.concat([idx[p] for p in rng.choice(pids, len(pids))])
        boots.append(pooled(sample, ["condition"]).set_index("condition")[metric])
    b = pd.DataFrame(boots)
    return pd.DataFrame({"estimate": point, "ci_low": b.quantile(0.025), "ci_high": b.quantile(0.975)})


def paired_differences(pm: pd.DataFrame, metrics: list[str], ref: str = "proposed",
                       n_boot: int = 2000, seed: int = 1) -> pd.DataFrame:
    """Per-participant paired differences (ref − baseline) with bootstrap CI
    and a Wilcoxon signed-rank test."""
    rng = np.random.default_rng(seed)
    wide = pm.pivot(index="participant_id", columns="condition", values=metrics)
    rows = []
    for m in metrics:
        for base in [c for c in wide[m].columns if c != ref]:
            d = (wide[m][ref] - wide[m][base]).dropna().to_numpy()
            if len(d) < 5:
                continue
            bs = [rng.choice(d, len(d)).mean() for _ in range(n_boot)]
            try:
                p = stats.wilcoxon(d).pvalue if np.any(d) else 1.0
            except ValueError:
                p = np.nan
            rows.append(dict(metric=m, comparison=f"{ref} − {base}", n=len(d), mean_diff=d.mean(),
                             ci_low=np.quantile(bs, 0.025), ci_high=np.quantile(bs, 0.975), wilcoxon_p=p))
    return pd.DataFrame(rows)


def group_tests(pm: pd.DataFrame, cohort: pd.DataFrame, condition: str = "proposed") -> pd.DataFrame:
    """Association between memory-related metadata and outcomes."""
    d = pm[pm.condition == condition].merge(cohort, on="participant_id")
    rows = []
    for metric in ["TSR", "independence", "FPR", "coverage"]:
        groups = [g[metric].dropna() for _, g in d.groupby("diagnosis")]
        h, p = stats.kruskal(*groups)
        rows.append(dict(test="Kruskal–Wallis by diagnosis", outcome=metric, statistic=h, p_value=p))
        for cov in ["moca", "prmq_prospective", "age", "digital_literacy"]:
            sub = d[[cov, metric]].dropna()
            rho, p = stats.spearmanr(sub[cov], sub[metric])
            rows.append(dict(test=f"Spearman ρ with {cov}", outcome=metric, statistic=rho, p_value=p))
        a, b = d[d.lives_alone][metric].dropna(), d[~d.lives_alone][metric].dropna()
        u, p = stats.mannwhitneyu(a, b)
        rows.append(dict(test="Mann–Whitney: lives alone vs not", outcome=metric, statistic=u, p_value=p))
    return pd.DataFrame(rows)


def logistic_completion(ep: pd.DataFrame, cohort: pd.DataFrame, condition: str = "proposed") -> pd.DataFrame:
    """Episode-level logistic regression of task completion on metadata
    (Newton–Raphson, no extra dependencies).  Odds ratios with Wald 95% CI;
    note that episodes are clustered within participants, so CIs are optimistic."""
    d = ep[ep.condition == condition].merge(cohort, on="participant_id")
    X = pd.DataFrame({
        "intercept": 1.0,
        "moca_per_point": d.moca - d.moca.mean(),
        "age_per_10y": (d.age - d.age.mean()) / 10,
        "lives_alone": d.lives_alone.astype(float),
        "sensors_connected": d.sensors_connected.astype(float),
        "digital_literacy": d.digital_literacy - d.digital_literacy.mean(),
        "safety_critical_task": d.safety_critical.astype(float),
    })
    y = d.completed_true.astype(float).to_numpy()
    Xv = X.to_numpy()
    beta = np.zeros(Xv.shape[1])
    for _ in range(50):
        mu = 1 / (1 + np.exp(-Xv @ beta))
        W = mu * (1 - mu)
        H = Xv.T @ (Xv * W[:, None])
        step = np.linalg.solve(H, Xv.T @ (y - mu))
        beta += step
        if np.abs(step).max() < 1e-8:
            break
    se = np.sqrt(np.diag(np.linalg.inv(H)))
    z = beta / se
    return pd.DataFrame({
        "term": X.columns, "OR": np.exp(beta), "ci_low": np.exp(beta - 1.96 * se),
        "ci_high": np.exp(beta + 1.96 * se), "p_value": 2 * stats.norm.sf(np.abs(z)),
    })


def recall_by_stratum(rc: pd.DataFrame) -> pd.DataFrame:
    g = rc.groupby(["stratum", "condition"]).agg(
        n=("question_id", "size"), coverage=("substantive", "mean"),
        appropriate=("appropriate", "mean"), fabricated=("fabricated", "sum"),
        overstated=("overstated", "sum"), consent_violations=("consent_violation", "sum"),
        stale=("stale_used", "sum"), wrong_event=("wrong_event", "sum"),
        released=("claims_released", "sum"), supported=("claims_supported", "sum"),
    )
    g["F_ext"] = g.supported / g.released.replace(0, np.nan)
    return g.reset_index()


def risk_coverage(rc: pd.DataFrame, condition: str = "proposed",
                  thetas: np.ndarray | None = None) -> pd.DataFrame:
    """Selective-answering curve for the release rule (Eq. 5): sweep theta_low
    over the recorded support scores S_min and recompute coverage C and F_ext.
    Gate-level abstentions (no record, conflict, permission) stay abstentions."""
    d = rc[rc.condition == condition]
    scored = d[d.s_min.notna()]
    thetas = np.round(np.linspace(0, 1, 101), 2) if thetas is None else thetas
    rows = []
    for th in thetas:
        rel = scored[scored.s_min >= th]
        rows.append(dict(theta_low=th, released=len(rel), coverage=len(rel) / len(d),
                         F_ext=rel.cand_supported.mean() if len(rel) else np.nan))
    return pd.DataFrame(rows)


def calibrate_threshold(rc: pd.DataFrame, target_f_ext: float = 0.99, dev_share: float = 0.5,
                        seed: int = 0, condition: str = "proposed") -> dict:
    """Pick the lowest theta_low reaching the target F_ext on development
    participants, then report it on held-out participants (Section 4.4)."""
    rng = np.random.default_rng(seed)
    pids = rc.participant_id.unique()
    dev = set(rng.choice(pids, int(len(pids) * dev_share), replace=False))
    is_dev = rc.participant_id.isin(dev)
    cd, ct = risk_coverage(rc[is_dev], condition), risk_coverage(rc[~is_dev], condition)
    ok = cd[cd.F_ext >= target_f_ext]
    th = float(ok.theta_low.min()) if len(ok) else 1.0
    pick = lambda c: c.loc[(c.theta_low - th).abs().idxmin()]
    return dict(theta_low=th, dev=pick(cd).to_dict(), test=pick(ct).to_dict(), curve_dev=cd, curve_test=ct)


In [ ]:
%%writefile pgmem_project/pgmem/schema.py
"""Dataset metadata (Frictionless Data Package) and validation.

The same descriptor documents the synthetic data and lets a team check
that a real, ethically approved dataset with the same structure can be
analysed by this code without changes.
"""
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

F = lambda name, type_, desc, **c: {"name": name, "type": type_, "description": desc,
                                     **({"constraints": c} if c else {})}

RESOURCES = {
    "participants": ("Cohort participants and memory-related metadata (one row = one person)", [
        F("participant_id", "string", "Pseudonymised identifier", required=True, unique=True, pattern=r"P\d{3}"),
        F("age", "integer", "Age, years", minimum=18, maximum=110),
        F("age_band", "string", "Age band", enum=["<50", "50-64", "65-74", "75-84", "85+"]),
        F("sex", "string", "Sex", enum=["F", "M"]),
        F("diagnosis", "string", "MCI — mild cognitive impairment; mild_dementia — mild dementia; ABI — acquired brain injury",
          enum=["MCI", "mild_dementia", "ABI"]),
        F("cdr_global", "number", "Clinical Dementia Rating, global score (empty for ABI)", minimum=0, maximum=3),
        F("moca", "integer", "Montreal Cognitive Assessment, education-adjusted (0–30)", minimum=0, maximum=30),
        F("education_years", "integer", "Years of education", minimum=0, maximum=30),
        F("prmq_prospective", "integer", "PRMQ prospective subscale (8–40, higher = more frequent failures)", minimum=8, maximum=40),
        F("prmq_retrospective", "integer", "PRMQ retrospective subscale (8–40)", minimum=8, maximum=40),
        F("months_since_diagnosis", "integer", "Months since diagnosis", minimum=0),
        F("lives_alone", "boolean", "Lives alone"),
        F("has_trusted_person", "boolean", "Has an agreed trusted person (R8)"),
        F("trusted_relation", "string", "Relation of the trusted person"),
        F("device", "string", "Device", enum=["smartphone", "tablet", "both"]),
        F("digital_literacy", "integer", "Self-rated digital literacy (1–5)", minimum=1, maximum=5),
        F("uses_voice", "boolean", "Uses voice interaction (R6)"),
        F("sensors_connected", "boolean", "Sensors connected (pill box, step counter)"),
        F("call_log_connected", "boolean", "Call log connected"),
        F("language", "string", "Interface language", enum=["uk", "en"]),
        F("agrees_to_level_reduction", "boolean", "Agrees to lowering the starting assistance level (Eq. 3)"),
        F("ability_latent", "number", "SIMULATION ONLY: latent ability from which responses are generated"),
    ]),
    "episodes": ("Scheduled task episodes (routines, R1–R4)", [
        F("episode_id", "string", "Episode identifier", required=True, unique=True),
        F("participant_id", "string", "Reference to participants", required=True),
        F("day", "integer", "Observation day (0–27)", minimum=0),
        F("task_type", "string", "Task type",
          enum=["medication_morning", "daily_walk", "appointment_prep", "bill_payment", "call_family"]),
        F("category", "string", "Data category used for consent"),
        F("safety_critical", "boolean", "Safety-critical task"),
        F("sensorable", "boolean", "Completion can be detected by a sensor"),
        F("scheduled", "datetime", "Scheduled time (ISO 8601)"),
    ]),
    "records": ("Personal event base: records r_i = (c, t, s, κ, σ, v, e), Equation (1)", [
        F("record_id", "string", "Record identifier", required=True, unique=True),
        F("person_id", "string", "Whose record this is", required=True),
        F("subject", "string", "Event identity"),
        F("content", "string", "c_i — content"),
        F("time", "datetime", "t_i — time"),
        F("source", "string", "s_i — source/provenance (calendar, user_voice, sensor, call_log, carer_entry)"),
        F("kappa", "number", "κ_i — confidence indicator (not a probability)", minimum=0, maximum=1),
        F("scope", "string", "σ_i — permitted recipients (comma-separated)"),
        F("version", "integer", "v_i — version", minimum=1),
        F("status", "string", "e_i — evidence status",
          enum=["plan", "self_report", "observation", "inference", "disputed"]),
        F("category", "string", "Data category"),
        F("kind", "string", "What the record speaks to: scheduled / occurred / content", enum=["scheduled", "occurred", "content"]),
        F("value", "string", "Normalised claim value"),
        F("supersedes", "string", "Previous version superseded by this record"),
        F("active", "boolean", "Current version"),
        F("disputed", "boolean", "Marked as disputed"),
    ]),
    "questions": ("Questions to the assistant about the past, with evaluation strata", [
        F("question_id", "string", "Identifier", required=True, unique=True),
        F("person_id", "string", "Whom the question is about"),
        F("stratum", "string", "Stratum", enum=["answerable_plan", "attendance", "self_report", "unanswerable",
                                                 "answerable_observation", "contradictory", "stale",
                                                 "answerable_note", "consent_restricted"]),
        F("requester", "string", "Who asks (the person or trusted:<id>)"),
        F("subject", "string", "Event"),
        F("kind", "string", "Claim kind"),
        F("category", "string", "Data category"),
        F("ideal", "string", "Reference (ideal) behaviour", enum=["answer", "abstain", "clarify", "refuse"]),
        F("text", "string", "Question text"),
    ]),
    "reference_history": ("Independent reference history (ground truth) for blinded claim scoring", [
        F("person_id", "string", "Person"),
        F("subject", "string", "Event"),
        F("kind", "string", "scheduled / occurred / reported / content"),
        F("value", "string", "True value"),
    ]),
}


def descriptor(seed: int, n: int, days: int) -> dict:
    return {
        "name": "pgmem-synthetic-memory-support",
        "title": "Synthetic dataset for evaluating a provenance-governed memory assistant",
        "version": "1.0.0",
        "licenses": [{"name": "CC-BY-4.0"}],
        "description": (
            "Fully synthetic data generated from the Prokopovych-Tkachenko framework 'Provenance-governed agentic AI "
            "for personal knowledge and memory support' (stage 1 of the evaluation protocol). No record "
            "corresponds to a real person; distributions are design assumptions, not empirical estimates."),
        "synthetic": True,
        "generator": {"package": "pgmem", "seed": seed, "participants": n, "days": days},
        "resources": [
            {"name": k, "path": f"{k}.csv", "format": "csv", "description": d, "schema": {"fields": f}}
            for k, (d, f) in RESOURCES.items()
        ],
    }


def write_descriptor(path: Path, seed: int, n: int, days: int) -> None:
    path.write_text(json.dumps(descriptor(seed, n, days), ensure_ascii=False, indent=2), encoding="utf-8")


def validate(data_dir: Path) -> list[str]:
    """Check every CSV against datapackage.json; returns a list of problems."""
    desc = json.loads((Path(data_dir) / "datapackage.json").read_text(encoding="utf-8"))
    problems = []
    for res in desc["resources"]:
        df = pd.read_csv(Path(data_dir) / res["path"], keep_default_na=False, na_values=[""])
        for f in res["schema"]["fields"]:
            name, c = f["name"], f.get("constraints", {})
            if name not in df.columns:
                problems.append(f"{res['name']}: missing column {name}")
                continue
            col = df[name]
            if c.get("required") and col.isna().any():
                problems.append(f"{res['name']}.{name}: empty values")
            if c.get("unique") and col.duplicated().any():
                problems.append(f"{res['name']}.{name}: duplicates")
            if "enum" in c and not col.dropna().astype(str).isin(c["enum"]).all():
                problems.append(f"{res['name']}.{name}: values outside enum")
            num = pd.to_numeric(col, errors="coerce")
            if "minimum" in c and (num < c["minimum"]).any():
                problems.append(f"{res['name']}.{name}: < {c['minimum']}")
            if "maximum" in c and (num > c["maximum"]).any():
                problems.append(f"{res['name']}.{name}: > {c['maximum']}")
            if "pattern" in c and not col.astype(str).str.fullmatch(c["pattern"]).all():
                problems.append(f"{res['name']}.{name}: does not match pattern")
    return problems


In [ ]:
%%writefile pgmem_project/pgmem/plots.py
"""Static report figures (matplotlib).

Design rules used throughout: each entity keeps one colour in every figure
(conditions: blue / orange / aqua; diagnoses: violet / magenta / green),
"higher is better" axes are oriented so that outward/up always means better,
violations of the invariants are shown as counts next to a tick or a cross,
never by colour alone, and grids and spines stay recessive.
"""
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib.colors import LinearSegmentedColormap  # noqa: E402
from matplotlib.patches import Patch  # noqa: E402

COND_ORDER = ["reminder", "single_agent", "proposed"]
COND_COLORS = {"proposed": "#2a78d6", "single_agent": "#eb6834", "reminder": "#1baf7a"}
COND_LABELS = {"reminder": "Reminder app", "single_agent": "Single agent + RAG", "proposed": "Proposed framework"}
DX_ORDER = ["MCI", "mild_dementia", "ABI"]
DX_COLORS = {"MCI": "#6250d6", "mild_dementia": "#e87ba4", "ABI": "#008300"}
DX_LABELS = {"MCI": "MCI", "mild_dementia": "Mild dementia", "ABI": "Acquired brain injury"}
INK, INK2, MUTED, SURFACE = "#0b0b0b", "#52514e", "#8a8984", "#fcfcfb"
BLUES = LinearSegmentedColormap.from_list("pg_blues", ["#eef4fc", "#9cc2ee", "#2a78d6", "#123f78"])
REDS = LinearSegmentedColormap.from_list("pg_reds", ["#fdf0ef", "#f2a3a2", "#e34948", "#8c1d1c"])
DIVERGE = LinearSegmentedColormap.from_list("pg_div", ["#d95926", "#f3c6b1", "#e9e8e4", "#a9c9ef", "#2a78d6"])

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 160, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c9c8c2",
    "axes.grid": True, "grid.color": "#e4e3de", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "font.size": 9.5, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "legend.frameon": False,
})


def _save(fig, out: Path | None, name: str):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)   # colorbar axes vs tight_layout
        fig.tight_layout()
    if out is not None:
        fig.savefig(Path(out) / name, bbox_inches="tight", facecolor=SURFACE)
    return fig


def _kde(x: np.ndarray, grid: np.ndarray, bw: float | None = None) -> np.ndarray:
    x = np.asarray(x, float)
    bw = bw or 1.06 * x.std() * len(x) ** -0.2 or 1.0
    return np.exp(-0.5 * ((grid[:, None] - x[None, :]) / bw) ** 2).sum(1) / (len(x) * bw * np.sqrt(2 * np.pi))


# ------------------------------------------------------------------ cohort
def cohort_overview(cohort: pd.DataFrame, out: Path | None = None):
    """Cohort fingerprint: age ridgelines, MoCA x PRMQ landscape with density
    contours, and a standardised metadata heat-strip per diagnosis."""
    fig = plt.figure(figsize=(14, 4.4))
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1.15, 1.25], wspace=0.35)

    ax = fig.add_subplot(gs[0])
    grid = np.linspace(15, 100, 300)
    for i, dx in enumerate(DX_ORDER[::-1]):
        g = cohort[cohort.diagnosis == dx]
        y = _kde(g.age.to_numpy(), grid)
        y = y / y.max() * 0.9 + i
        ax.fill_between(grid, i, y, color=DX_COLORS[dx], alpha=0.35, lw=0)
        ax.plot(grid, y, color=DX_COLORS[dx], lw=2)
        med = g.age.median()
        ax.plot([med, med], [i, i + 0.9], color=INK, lw=1, ls=":")
        ax.text(16 if med > 60 else 72, i + 0.55, f"{DX_LABELS[dx]}\nn={len(g)}, median {med:.0f}", fontsize=8,
                color=INK2)
    ax.set(yticks=[], xlabel="Age, years", title="Age by diagnosis (ridgelines)")
    ax.grid(axis="y", visible=False)

    ax = fig.add_subplot(gs[1])
    jit = np.random.default_rng(0)
    for dx in DX_ORDER:
        g = cohort[cohort.diagnosis == dx]
        xm, ym = g.moca.to_numpy(float), g.prmq_prospective.to_numpy(float)
        xx, yy = np.meshgrid(np.linspace(8, 31, 80), np.linspace(8, 41, 80))
        bx, by = 1.06 * xm.std() * len(xm) ** -0.2, 1.06 * ym.std() * len(ym) ** -0.2
        zz = np.exp(-0.5 * (((xx[..., None] - xm) / bx) ** 2 + ((yy[..., None] - ym) / by) ** 2)).sum(-1)
        ax.contour(xx, yy, zz, levels=[zz.max() * 0.35, zz.max() * 0.7], colors=DX_COLORS[dx], linewidths=[1, 2])
        ax.scatter(xm + jit.uniform(-.3, .3, len(xm)), ym + jit.uniform(-.3, .3, len(ym)), s=10, alpha=0.45,
                   color=DX_COLORS[dx], edgecolors="none", label=DX_LABELS[dx])
    ax.axvspan(26, 31, color="#e9e8e4", alpha=0.6, lw=0)
    ax.text(26.3, 39, "MoCA ≥ 26\n(normal range)", fontsize=7.5, color=INK2, va="top")
    ax.set(xlim=(8, 31), ylim=(8, 41), xlabel="MoCA (0–30, higher = better)",
           ylabel="PRMQ prospective (higher = more failures)", title="Cognitive landscape")
    ax.legend(loc="lower left", fontsize=8, markerscale=2)

    ax = fig.add_subplot(gs[2])
    cols = {"age": "Age", "moca": "MoCA", "prmq_prospective": "PRMQ-P", "prmq_retrospective": "PRMQ-R",
            "education_years": "Education", "digital_literacy": "Digital literacy", "lives_alone": "Lives alone",
            "has_trusted_person": "Trusted person", "sensors_connected": "Sensors", "uses_voice": "Voice"}
    c = cohort[list(cols)].astype(float)
    z = (c.groupby(cohort.diagnosis).mean() - c.mean()) / c.std()
    z = z.reindex(DX_ORDER)
    im = ax.imshow(z.to_numpy(), cmap=DIVERGE, vmin=-1.2, vmax=1.2, aspect="auto")
    for (i, j), v in np.ndenumerate(z.to_numpy()):
        ax.text(j, i, f"{v:+.1f}", ha="center", va="center", fontsize=8, color=INK)
    ax.set_xticks(range(len(cols)), list(cols.values()), rotation=40, ha="right")
    ax.set_yticks(range(3), ["MCI", "Mild\ndementia", "ABI"])
    ax.grid(False)
    ax.set_title("Metadata fingerprint (z-score vs cohort)")
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="SD from cohort mean")
    return _save(fig, out, "fig1_cohort.png")


# ------------------------------------------------------------- conditions
RADAR_AXES = [  # (metric, label, transform to "higher is better" in [0, 1])
    ("TSR", "Task success", lambda v: v),
    ("FPR", "1 − false prompts", lambda v: 1 - v),
    ("independence", "Independence\n(L0–L1)", lambda v: v),
    ("invented_outcome_rate", "No invented\noutcomes", lambda v: 1 - v),
    ("F_ext", "Recall\nfaithfulness", lambda v: v),
    ("coverage", "Answer\ncoverage", lambda v: v),
    ("overstated_per_100", "No overstated\nclaims", lambda v: 1 - min(v, 100) / 100),
    ("appropriate_behaviour", "Appropriate\nbehaviour", lambda v: v),
]


def condition_radar(summary: pd.DataFrame, out: Path | None = None, name: str = "fig2_radar.png"):
    """Polar profile of all headline indicators with bootstrap-CI bands; every
    axis is oriented so that the outer ring is the desirable end."""
    ang = np.linspace(0, 2 * np.pi, len(RADAR_AXES), endpoint=False)
    fig, ax = plt.subplots(figsize=(7.2, 6.4), subplot_kw=dict(polar=True))
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    for c in COND_ORDER:
        vals, lo, hi, undefined = [], [], [], []
        for k, (m, _, f) in enumerate(RADAR_AXES):
            row = summary[m].loc[c]
            if pd.isna(row.estimate):
                vals.append(0.0); lo.append(0.0); hi.append(0.0); undefined.append(k)
                continue
            a, b = sorted([f(row.ci_low), f(row.ci_high)])
            vals.append(f(row.estimate)); lo.append(a); hi.append(b)
        close = lambda v: np.r_[v, v[:1]]
        th = close(ang)
        ax.fill_between(th, close(np.array(lo)), close(np.array(hi)), color=COND_COLORS[c], alpha=0.25, lw=0)
        ax.plot(th, close(np.array(vals)), color=COND_COLORS[c], lw=2, label=COND_LABELS[c])
        ax.scatter(ang, vals, s=28, color=COND_COLORS[c], edgecolor=SURFACE, linewidth=1.5, zorder=3)
        for k in undefined:
            ax.scatter(ang[k], 0.04, marker="x", s=40, color=COND_COLORS[c], zorder=4)
    ax.set_xticks(ang, [lbl for _, lbl, _ in RADAR_AXES], fontsize=8.5)
    ax.tick_params(axis="x", pad=14)
    ax.set_rlabel_position(22.5)
    ax.set_ylim(0, 1.02)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0], ["0.25", "0.50", "0.75", "1.00"], fontsize=7, color=MUTED)
    ax.spines["polar"].set_visible(False)
    ax.set_title("Condition profiles — outer ring is better\n(bands: 95% cluster-bootstrap CI; × = undefined)",
                 loc="center", pad=22)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8.5)
    return _save(fig, out, name)


def condition_dots(summary: pd.DataFrame, metrics: list[str], out: Path | None = None,
                   name: str = "fig2b_dots.png"):
    """Dot-and-whisker small multiples (estimate + 95% CI); replaces bar charts
    so that intervals, not bar areas, carry the comparison."""
    fig, axes = plt.subplots(1, len(metrics), figsize=(3.1 * len(metrics), 2.9), sharey=True)
    for ax, m in zip(np.atleast_1d(axes), metrics):
        s = summary[m].reindex(COND_ORDER)
        for i, c in enumerate(COND_ORDER):
            e, lo, hi = s.loc[c]
            if pd.isna(e):
                ax.text(0.02, i, "undefined", va="center", fontsize=8, color=MUTED, transform=ax.get_yaxis_transform())
                continue
            ax.plot([lo, hi], [i, i], color=COND_COLORS[c], lw=2, solid_capstyle="round")
            ax.scatter([e], [i], s=60, color=COND_COLORS[c], edgecolor=SURFACE, linewidth=2, zorder=3)
            ax.annotate(f"{e:.2f}", (e, i), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8,
                        color=INK2)
        ax.set_title(m, fontsize=10)
        ax.set_ylim(-0.6, 2.6)
        ax.grid(axis="y", visible=False)
    np.atleast_1d(axes)[0].set_yticks(range(3), [COND_LABELS[c] for c in COND_ORDER])
    return _save(fig, out, name)


def paired_forest(paired: pd.DataFrame, out: Path | None = None):
    """Forest plot of per-participant paired differences (proposed − baseline)."""
    d = paired.iloc[::-1].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 0.42 * len(d) + 1.2))
    for i, r in d.iterrows():
        base = r.comparison.split("− ")[-1]
        col = COND_COLORS.get(base, INK2)
        ax.plot([r.ci_low, r.ci_high], [i, i], color=col, lw=2.2, solid_capstyle="round")
        ax.scatter([r.mean_diff], [i], marker="D", s=40, color=col, edgecolor=SURFACE, lw=1.5, zorder=3)
        ax.text(1.02, i, f"{r.mean_diff:+.3f}  [{r.ci_low:+.3f}, {r.ci_high:+.3f}]", va="center", fontsize=8,
                color=INK2, transform=ax.get_yaxis_transform())
    ax.axvline(0, color=INK, lw=0.8)
    ax.set_yticks(range(len(d)), [f"{r.metric}  vs {COND_LABELS[r.comparison.split('− ')[-1]]}"
                                  for _, r in d.iterrows()], fontsize=8.5)
    ax.grid(axis="y", visible=False)
    ax.set(xlabel="Mean paired difference, proposed − baseline (95% bootstrap CI)",
           title="Paired differences per participant")
    return _save(fig, out, "fig3_forest.png")


# --------------------------------------------------------- people x outcome
def tsr_raincloud(pm: pd.DataFrame, cohort: pd.DataFrame, out: Path | None = None):
    """Raincloud plots (half-violin + box + jittered rain) of participant TSR,
    and the MoCA–independence relation with a LOWESS-like binned trend."""
    d = pm.merge(cohort[["participant_id", "diagnosis", "moca"]], on="participant_id")
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw=dict(width_ratios=[1.6, 1]))
    rng = np.random.default_rng(1)
    w = 0.27
    for i, dx in enumerate(DX_ORDER):
        for j, c in enumerate(COND_ORDER):
            v = d[(d.condition == c) & (d.diagnosis == dx)].TSR.dropna().to_numpy()
            x0 = i + (j - 1) * w
            grid = np.linspace(v.min() - .02, v.max() + .02, 120)
            k = _kde(v, grid)
            k = k / k.max() * w * 0.45
            ax[0].fill_betweenx(grid, x0, x0 + k, color=COND_COLORS[c], alpha=0.45, lw=0)
            ax[0].boxplot([v], positions=[x0], widths=w * 0.18, showfliers=False, patch_artist=True,
                          boxprops=dict(facecolor=SURFACE, edgecolor=INK2), medianprops=dict(color=INK, lw=1.5),
                          whiskerprops=dict(color=INK2), capprops=dict(color=INK2))
            ax[0].scatter(x0 - w * 0.25 + rng.uniform(-w * .12, w * .12, len(v)), v, s=6, alpha=0.5,
                          color=COND_COLORS[c], edgecolors="none")
    ax[0].set_xticks(range(3), [DX_LABELS[d] for d in DX_ORDER])
    ax[0].set(ylabel="Participant TSR", title="Task success by diagnosis and condition (raincloud)")
    ax[0].legend(handles=[Patch(color=COND_COLORS[c], label=COND_LABELS[c]) for c in COND_ORDER],
                 loc="lower right", fontsize=8, ncol=1)
    ax[0].grid(axis="x", visible=False)

    p = d[d.condition == "proposed"]
    for dx in DX_ORDER:
        g = p[p.diagnosis == dx]
        ax[1].scatter(g.moca + rng.uniform(-.25, .25, len(g)), g.independence, s=14, alpha=0.6,
                      color=DX_COLORS[dx], edgecolors="none", label=DX_LABELS[dx])
    b = p.groupby(p.moca.clip(12, 30)).independence.mean()
    ax[1].plot(b.index, b.rolling(3, center=True, min_periods=1).mean(), color=INK, lw=2, label="Binned trend")
    ax[1].set(xlabel="MoCA", ylabel="Share of completions at L0–L1",
              title="Independence vs MoCA (proposed)")
    ax[1].legend(fontsize=8, loc="lower right")
    return _save(fig, out, "fig4_raincloud.png")


# ------------------------------------------------------------------ recall
def recall_heatmap(strata: pd.DataFrame, out: Path | None = None):
    """Strata x condition matrix: colour = share of reference behaviour; the
    cell text adds the invariant violations hidden behind that share."""
    piv = strata.pivot(index="stratum", columns="condition", values="appropriate").reindex(columns=COND_ORDER)
    order = piv.mean(axis=1).sort_values().index
    piv = piv.loc[order]
    st = strata.set_index(["stratum", "condition"])
    fig, ax = plt.subplots(figsize=(8.6, 0.5 * len(piv) + 1.6))
    ax.imshow(piv.to_numpy(), cmap=BLUES, vmin=0, vmax=1, aspect="auto")
    for (i, j), v in np.ndenumerate(piv.to_numpy()):
        r = st.loc[(piv.index[i], piv.columns[j])]
        flags = [f"{int(r[k])} {lbl}" for k, lbl in [("fabricated", "fab"), ("overstated", "over"),
                                                       ("consent_violations", "consent"), ("stale", "stale"),
                                                       ("wrong_event", "wrong-ev")] if r[k] > 0]
        ax.text(j, i - 0.12, f"{v:.0%}", ha="center", va="center", fontsize=9.5, fontweight="bold",
                color="white" if v > 0.6 else INK)
        ax.text(j, i + 0.22, "✗ " + ", ".join(flags) if flags else "✓ no violations", ha="center", va="center",
                fontsize=7, color="white" if v > 0.6 else INK2)
    ax.set_xticks(range(3), [COND_LABELS[c] for c in COND_ORDER])
    ax.xaxis.tick_top()
    ax.set_yticks(range(len(piv)), [s.replace("_", " ") for s in piv.index])
    ax.grid(False)
    ax.set_title("Recall: share of reference behaviour and invariant violations per stratum", pad=28)
    return _save(fig, out, "fig5_recall_heatmap.png")


def invariant_scoreboard(ep: pd.DataFrame, rc: pd.DataFrame, out: Path | None = None):
    """One row per invariant counter; zero cells get a tick, non-zero cells a
    red intensity proportional to log(count) and the exact number."""
    e = ep.groupby("condition")[["unauthorised_contact", "wrong_failure", "invented_outcome"]].sum()
    r = rc.groupby("condition")[["consent_violation", "overstated", "stale_used", "fabricated"]].sum()
    tab = e.join(r).reindex(COND_ORDER)
    labels = {"unauthorised_contact": "I1 contact after consent withdrawal",
              "consent_violation": "I1 disclosure outside consent scope",
              "overstated": "I2 claim stronger than its record",
              "fabricated": "I2 fabricated recollection",
              "invented_outcome": "I3 outcome inferred from silence",
              "wrong_failure": "I3 completed task logged as missed",
              "stale_used": "I4 answer from superseded version"}
    tab = tab[list(labels)].T
    applicable = pd.DataFrame(True, index=tab.index, columns=tab.columns)
    applicable.loc[["consent_violation", "overstated", "fabricated", "stale_used"], "reminder"] = False
    applicable.loc["unauthorised_contact", "reminder"] = False
    z = np.log1p(tab.to_numpy(float))
    fig, ax = plt.subplots(figsize=(8.6, 4.0))
    ax.imshow(np.where(applicable & (tab.to_numpy() > 0), z, np.nan), cmap=REDS, vmin=0, vmax=max(z.max(), 1), aspect="auto")
    for (i, j), v in np.ndenumerate(tab.to_numpy()):
        if not applicable.iloc[i, j]:
            txt, col = "n/a", MUTED
        elif v == 0:
            txt, col = "✓ 0", "#008300"
        else:
            txt, col = f"✗ {int(v)}", "white" if z[i, j] > 0.6 * z.max() else INK
        ax.text(j, i, txt, ha="center", va="center", fontsize=9.5, fontweight="bold", color=col)
    ax.set_xticks(range(3), [COND_LABELS[c] for c in COND_ORDER])
    ax.xaxis.tick_top()
    ax.set_yticks(range(len(tab)), [labels[k] for k in tab.index])
    ax.grid(False)
    ax.set_title("Invariant scoreboard (counts over all episodes and questions)", pad=28)
    return _save(fig, out, "fig6_invariants.png")


def risk_coverage(curve: pd.DataFrame, theta_low: float, theta_high: float, out: Path | None = None,
                  calibrated: dict | None = None):
    """Selective-answering curve: what the release rule (Eq. 5) trades as
    theta_low moves. The current operating point and the calibrated point
    (dev split -> held-out split) are marked."""
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.0))
    c = curve.dropna()
    ax[0].plot(c.coverage, c.F_ext, color=COND_COLORS["proposed"], lw=2)
    sc = ax[0].scatter(c.coverage, c.F_ext, c=c.theta_low, cmap=BLUES, s=14, zorder=3, edgecolors="none")
    pt = curve.loc[(curve.theta_low - theta_low).abs().idxmin()]
    ax[0].scatter([pt.coverage], [pt.F_ext], s=140, facecolor="none", edgecolor=INK, lw=1.5, zorder=4)
    ax[0].annotate(f"θ_low = {theta_low:.2f}\nC = {pt.coverage:.3f}, F_ext = {pt.F_ext:.3f}",
                   (pt.coverage, pt.F_ext), xytext=(-170, 60), textcoords="offset points", fontsize=8,
                   arrowprops=dict(arrowstyle="-", color=INK2))
    if calibrated:
        t = calibrated["test"]
        ax[0].scatter([t["coverage"]], [t["F_ext"]], marker="*", s=180, color="#eda100", edgecolor=INK, zorder=5,
                      label=f"calibrated θ_low={calibrated['theta_low']:.2f} (held-out)")
        ax[0].legend(fontsize=8, loc="lower left")
    fig.colorbar(sc, ax=ax[0], label="θ_low")
    ax[0].set(xlabel="Answer coverage C", ylabel="Recall faithfulness F_ext",
              title="Risk–coverage curve of the release rule")
    ax[1].plot(curve.theta_low, curve.coverage, color=COND_COLORS["proposed"], lw=2, label="coverage C")
    ax[1].plot(curve.theta_low, curve.F_ext, color=INK, lw=2, ls="--", label="F_ext")
    ax[1].axvspan(0, theta_low, color="#f2a3a2", alpha=0.25, lw=0)
    ax[1].axvspan(theta_low, theta_high, color="#f6d58a", alpha=0.3, lw=0)
    ax[1].axvspan(theta_high, 1, color="#9cc2ee", alpha=0.3, lw=0)
    for x, t in [(theta_low / 2, "abstain"), ((theta_low + theta_high) / 2, "answer with\nuncertainty"),
                 ((theta_high + 1) / 2, "answer with\nsource")]:
        ax[1].text(x, 0.06, t, ha="center", fontsize=8, color=INK2)
    ax[1].set(xlabel="θ_low", ylim=(0, 1.04), title="Equation (5) zones over the support score")
    ax[1].legend(fontsize=8, loc="center left", bbox_to_anchor=(0.0, 0.72))
    return _save(fig, out, "fig7_risk_coverage.png")


# ------------------------------------------------------------- assistance
def level_distribution(ep: pd.DataFrame, out: Path | None = None):
    """Diverging stacked bars: completions at L0–L1 extend right, at L2–L4
    left, non-completions are reported separately — so 'more help' and
    'independent' read as opposite directions."""
    d = ep.copy()
    d["lvl"] = np.where(d.completed_true, d.completion_level.fillna(-1).astype(int), -1)
    tab = pd.crosstab(d.condition, d.lvl, normalize="index").reindex(COND_ORDER).fillna(0)
    for k in [-1, 0, 1, 2, 3, 4]:
        if k not in tab:
            tab[k] = 0.0
    colors = {0: "#123f78", 1: "#2a78d6", 2: "#eda100", 3: "#eb6834", 4: "#b43a14"}
    fig, ax = plt.subplots(figsize=(10, 3.2))
    y = np.arange(len(tab))
    left = np.zeros(len(tab))
    for k in (0, 1):
        ax.barh(y, tab[k], left=left, color=colors[k], edgecolor=SURFACE, lw=2, height=0.6, label=f"L{k}")
        left += tab[k].to_numpy()
    right = np.zeros(len(tab))
    for k in (2, 3, 4):
        ax.barh(y, -tab[k], left=-right, color=colors[k], edgecolor=SURFACE, lw=2, height=0.6, label=f"L{k}")
        right += tab[k].to_numpy()
    for i, c in enumerate(tab.index):
        ax.text(left[i] + 0.01, i, f"{left[i]:.0%} at L0–L1", va="center", fontsize=8, color=INK2)
        ax.text(-right[i] - 0.01, i, f"{right[i]:.0%}", va="center", ha="right", fontsize=8, color=INK2)
        ax.text(1.02, i, f"not completed {tab.loc[c, -1]:.0%}", va="center", fontsize=8, color=MUTED,
                transform=ax.get_yaxis_transform())
    ax.axvline(0, color=INK, lw=0.8)
    ax.set_yticks(y, [COND_LABELS[c] for c in tab.index])
    ax.set_xlim(-0.45, 1.0)
    ax.set_xticks([-0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8], ["40%", "20%", "0", "20%", "40%", "60%", "80%"])
    ax.grid(axis="y", visible=False)
    ax.set(title="Assistance level at completion  ◀ more help · independent ▶", xlabel="share of episodes")
    ax.legend(ncol=5, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.25))
    return _save(fig, out, "fig8_levels.png")


def adaptation_over_time(ep: pd.DataFrame, out: Path | None = None):
    """Equation (3) at work: mean starting level per day in the proposed
    condition for non-safety-critical tasks, with the share of escalations."""
    p = ep[(ep.condition == "proposed")]
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
    task_colors = {"daily_walk": "#2a78d6", "call_family": "#eb6834", "appointment_prep": "#1baf7a",
                   "bill_payment": "#eda100", "medication_morning": "#8a8984"}
    flat = []
    for task, g in p.groupby("task_type"):
        s = g.groupby("day").start_level.mean()
        if s.nunique() == 1:
            flat.append(task.replace("_", " "))
            ax[0].plot(s.index, s.values, lw=2, color=MUTED, ls="--" if task == "medication_morning" else ":")
            continue
        ax[0].plot(s.index, s.values, lw=2, color=task_colors.get(task, INK2))
        ax[0].text(s.index[-1] + 0.4, s.values[-1], task.replace("_", " "), fontsize=8, va="center", color=INK2)
    if flat:
        ax[0].text(p.day.max() + 0.4, s.values[0] if s.nunique() == 1 else 1.0, "unchanged:\n" + "\n".join(flat),
                   fontsize=7.5, va="top", color=MUTED)
    ax[0].set(xlabel="day", ylabel="mean starting level", xlim=(0, p.day.max() + 7),
              title="Starting level adapts down (Eq. 3); safety-critical stays fixed")
    for c in COND_ORDER:
        s = ep[ep.condition == c].groupby("day").escalations.mean()
        ax[1].plot(s.index, s.rolling(3, min_periods=1).mean(), lw=2, color=COND_COLORS[c], label=COND_LABELS[c])
    ax[1].set(xlabel="day", ylabel="escalations per episode (3-day mean)", title="Escalation load over time")
    ax[1].legend(fontsize=8, loc="upper right", bbox_to_anchor=(1.0, 0.72))
    return _save(fig, out, "fig9_adaptation.png")


def entailment_heatmap(mat: pd.DataFrame, title: str, out: Path | None = None, name: str = "fig10_nli.png"):
    """Premise x hypothesis entailment probabilities from the NLI model."""
    fig, ax = plt.subplots(figsize=(1.6 * mat.shape[1] + 4, 0.55 * mat.shape[0] + 1.8))
    ax.imshow(mat.to_numpy(), cmap=BLUES, vmin=0, vmax=1, aspect="auto")
    for (i, j), v in np.ndenumerate(mat.to_numpy()):
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9, color="white" if v > 0.6 else INK)
    ax.set_xticks(range(mat.shape[1]), [_wrap(c, 22) for c in mat.columns], fontsize=8)
    ax.xaxis.tick_top()
    ax.set_yticks(range(mat.shape[0]), [_wrap(r, 48) for r in mat.index], fontsize=8)
    ax.grid(False)
    ax.set_title(title, pad=40)
    return _save(fig, out, name)


def _wrap(s: str, n: int) -> str:
    words, lines, cur = str(s).split(), [], ""
    for w in words:
        if len(cur) + len(w) + 1 > n and cur:
            lines.append(cur)
            cur = w
        else:
            cur = f"{cur} {w}".strip()
    return "\n".join(lines + [cur])


In [ ]:
%%writefile pgmem_project/pgmem/interactive.py
"""Interactive figures (Plotly) for the notebook and HTML reports.

Every figure has hover details; colours match ``plots`` so an entity looks
the same in static and interactive views.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from .plots import COND_COLORS, COND_LABELS, COND_ORDER, DX_COLORS, DX_LABELS, DX_ORDER, RADAR_AXES

FONT = dict(family="Inter, Helvetica, Arial, sans-serif", size=12, color="#0b0b0b")
LAYOUT = dict(template="plotly_white", font=FONT, paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb",
              margin=dict(l=40, r=30, t=70, b=40), hoverlabel=dict(font_size=12))


def _rgba(hex_: str, a: float) -> str:
    h = hex_.lstrip("#")
    return f"rgba({int(h[0:2], 16)},{int(h[2:4], 16)},{int(h[4:6], 16)},{a})"


def save(fig: go.Figure, out: Path | None, name: str) -> go.Figure:
    if out is not None:
        fig.write_html(Path(out) / name, include_plotlyjs="cdn")
    return fig


def episode_sankey(ep: pd.DataFrame, out: Path | None = None) -> go.Figure:
    """Condition -> what really happened -> what was recorded. The flow
    'not completed/completed -> missed (inferred)' is where silence becomes a
    fabricated outcome (I3); the proposed framework routes it to 'unknown'."""
    d = ep.copy()
    d["truth"] = np.where(~d.completed_true, "Not completed",
                          np.where(d.completion_level.fillna(0) <= 1, "Completed at L0–L1", "Completed at L2–L4"))
    rec_labels = {"completed": "Recorded: completed (with source)", "unknown": "Recorded: unknown",
                  "postponed": "Recorded: postponed", "declined": "Recorded: declined",
                  "missed_inferred": "Recorded: missed (inferred from silence)"}
    d["rec"] = d.recorded_state.map(rec_labels)
    nodes = [COND_LABELS[c] for c in COND_ORDER] + ["Completed at L0–L1", "Completed at L2–L4", "Not completed"] \
        + list(rec_labels.values())
    idx = {n: i for i, n in enumerate(nodes)}
    node_col = [COND_COLORS[c] for c in COND_ORDER] + ["#123f78", "#eda100", "#8a8984"] + \
               ["#1baf7a", "#c3c2b7", "#9cc2ee", "#9cc2ee", "#e34948"]
    src, tgt, val, col, hov = [], [], [], [], []
    for (c, t), n in d.groupby(["condition", "truth"]).size().items():
        src.append(idx[COND_LABELS[c]]); tgt.append(idx[t]); val.append(n)
        col.append(_rgba(COND_COLORS[c], 0.35)); hov.append(f"{COND_LABELS[c]} → {t}: {n:,} episodes")
    for (c, t, r), n in d.groupby(["condition", "truth", "rec"]).size().items():
        src.append(idx[t]); tgt.append(idx[r]); val.append(n)
        bad = "missed" in r and t != "Not completed"
        col.append("rgba(227,73,72,0.75)" if bad else _rgba(COND_COLORS[c], 0.3))
        hov.append(f"{COND_LABELS[c]}: {t} → {r}: {n:,}" + ("  ⚠ completed task logged as missed (I3)" if bad else ""))
    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(label=nodes, color=node_col, pad=18, thickness=16, line=dict(color="#fcfcfb", width=2)),
        link=dict(source=src, target=tgt, value=val, color=col, customdata=hov,
                  hovertemplate="%{customdata}<extra></extra>"),
    ))
    fig.update_layout(**LAYOUT, height=560,
                      title="Episode flow: condition → true outcome → recorded state "
                            "<br><sup>red ribbons = completed tasks recorded as missed (violates I3)</sup>")
    return save(fig, out, "sankey_episodes.html")


def recall_sunburst(rc: pd.DataFrame, out: Path | None = None) -> go.Figure:
    """Condition -> stratum -> decision, coloured by whether the decision
    matches the reference behaviour."""
    d = rc.copy()
    d["condition_label"] = d.condition.map(COND_LABELS)
    d["verdict"] = np.where(d.appropriate, "appropriate", "inappropriate")
    g = d.groupby(["condition_label", "stratum", "decision", "verdict"]).size().reset_index(name="n")
    fig = px.sunburst(g, path=["condition_label", "stratum", "decision"], values="n", color="verdict",
                      color_discrete_map={"appropriate": "#2a78d6", "inappropriate": "#e34948", "(?)": "#c3c2b7"})
    fig.update_traces(hovertemplate="%{id}<br>%{value:,} answers<extra></extra>", insidetextorientation="radial")
    fig.update_layout(**LAYOUT, height=640,
                      title="Recall decisions by condition and stratum<br><sup>blue = reference behaviour, "
                            "red = not; click a ring to zoom</sup>")
    return save(fig, out, "sunburst_recall.html")


def condition_radar(summary: pd.DataFrame, out: Path | None = None) -> go.Figure:
    fig = go.Figure()
    labels = [lbl.replace("\n", " ") for _, lbl, _ in RADAR_AXES]
    for c in COND_ORDER:
        r, txt = [], []
        for m, lbl, f in RADAR_AXES:
            row = summary[m].loc[c]
            ok = not pd.isna(row.estimate)
            r.append(f(row.estimate) if ok else 0)
            txt.append(f"{m} = {row.estimate:.3f} [{row.ci_low:.3f}, {row.ci_high:.3f}]" if ok else f"{m}: undefined")
        fig.add_trace(go.Scatterpolar(r=r + r[:1], theta=labels + labels[:1], name=COND_LABELS[c], fill="toself",
                                      line=dict(color=COND_COLORS[c], width=2), fillcolor=_rgba(COND_COLORS[c], .15),
                                      customdata=txt + txt[:1], hovertemplate="%{customdata}<extra>%{fullData.name}"
                                                                              "</extra>"))
    fig.update_layout(**LAYOUT, height=560, polar=dict(radialaxis=dict(range=[0, 1.02], tickfont_size=9)),
                      title="Condition profiles — outer ring is better (hover for raw value and 95% CI)")
    return save(fig, out, "radar_conditions.html")


def cohort_parallel(cohort: pd.DataFrame, pm: pd.DataFrame, out: Path | None = None) -> go.Figure:
    """Parallel coordinates: every line is one synthetic person, coloured by
    TSR under the proposed framework. Drag on an axis to filter (brush)."""
    p = pm[pm.condition == "proposed"][["participant_id", "TSR", "independence", "coverage"]]
    d = cohort.merge(p, on="participant_id")
    dx_code = d.diagnosis.map({k: i for i, k in enumerate(DX_ORDER)})
    dims = [dict(label="Diagnosis", values=dx_code, tickvals=[0, 1, 2], ticktext=[DX_LABELS[x] for x in DX_ORDER]),
            dict(label="Age", values=d.age), dict(label="MoCA", values=d.moca),
            dict(label="PRMQ-P", values=d.prmq_prospective), dict(label="Digital literacy", values=d.digital_literacy),
            dict(label="Lives alone", values=d.lives_alone.astype(int), tickvals=[0, 1], ticktext=["no", "yes"]),
            dict(label="Independence", values=d.independence, range=[0, 1]),
            dict(label="Coverage", values=d.coverage, range=[0, 1]),
            dict(label="TSR", values=d.TSR)]
    fig = go.Figure(go.Parcoords(line=dict(color=d.TSR, colorscale=[[0, "#e34948"], [0.5, "#e9e8e4"], [1, "#2a78d6"]],
                                           showscale=True, colorbar=dict(title="TSR")), dimensions=dims))
    fig.update_layout(**LAYOUT, height=520, title="Who benefits least? Metadata → outcome per participant "
                                                  "(drag along any axis to filter)")
    return save(fig, out, "parallel_cohort.html")


def weekly_trajectories(ep: pd.DataFrame, cohort: pd.DataFrame, out: Path | None = None) -> go.Figure:
    """Animated bubble chart: each bubble is a participant, x = MoCA,
    y = cumulative TSR up to that week, size = escalations so far."""
    d = ep.merge(cohort[["participant_id", "moca", "diagnosis"]], on="participant_id")
    d["week"] = d.day // 7 + 1
    g = d.groupby(["condition", "participant_id", "week"]).agg(
        done=("completed_true", "sum"), n=("episode_id", "size"), esc=("escalations", "sum"),
        moca=("moca", "first"), diagnosis=("diagnosis", "first")).reset_index()
    g = g.sort_values("week")
    for k in ("done", "n", "esc"):
        g[k] = g.groupby(["condition", "participant_id"])[k].cumsum()
    g["TSR"] = g.done / g.n
    g["Condition"] = g.condition.map(COND_LABELS)
    g["MoCA (jittered)"] = g.moca + np.random.default_rng(0).uniform(-.35, .35, len(g))
    g["Week"] = "week " + g.week.astype(str)
    fig = px.scatter(g, x="MoCA (jittered)", y="TSR", size="esc", color="Condition", animation_frame="Week",
                     animation_group="participant_id", hover_name="participant_id",
                     hover_data=dict(diagnosis=True, esc=True, moca=True, **{"MoCA (jittered)": False}),
                     color_discrete_map={COND_LABELS[c]: COND_COLORS[c] for c in COND_ORDER},
                     size_max=18, range_y=[0.3, 1.02], range_x=[8, 31], opacity=0.65)
    fig.update_layout(**LAYOUT, height=560,
                      title="Cumulative task success per participant, week by week (press ▶)<br>"
                            "<sup>bubble size = escalations so far</sup>")
    return save(fig, out, "animated_weeks.html")


def risk_coverage(curve: pd.DataFrame, theta_low: float, out: Path | None = None) -> go.Figure:
    c = curve.dropna()
    fig = go.Figure(go.Scatter(x=c.coverage, y=c.F_ext, mode="lines+markers",
                               marker=dict(color=c.theta_low, colorscale="Blues", size=7, showscale=True,
                                           colorbar=dict(title="θ_low")),
                               line=dict(color="#2a78d6", width=2), customdata=c[["theta_low", "released"]],
                               hovertemplate="θ_low=%{customdata[0]:.2f}<br>coverage=%{x:.3f}<br>F_ext=%{y:.3f}"
                                             "<br>released=%{customdata[1]}<extra></extra>"))
    pt = curve.loc[(curve.theta_low - theta_low).abs().idxmin()]
    fig.add_trace(go.Scatter(x=[pt.coverage], y=[pt.F_ext], mode="markers", name="current θ_low",
                             marker=dict(size=16, symbol="circle-open", color="#0b0b0b", line_width=2)))
    fig.update_layout(**LAYOUT, height=440, showlegend=False, xaxis_title="Answer coverage C",
                      yaxis_title="Recall faithfulness F_ext", title="Risk–coverage of the release rule (Eq. 5)")
    return save(fig, out, "risk_coverage.html")


In [ ]:
%%writefile pgmem_project/pgmem/architecture.py
"""Diagrams of a deployable neural-network realisation of the framework.

The paper's architecture (Figure 1) names logical roles. These diagrams bind
each role to concrete, openly available components so the design can be
built and audited:

* ``system_architecture``   – layers, devices, models, stores, governance.
* ``neural_recall_pipeline`` – the grounded-recall data flow (Eqs. 4–5) with
  model sizes and tensor shapes; neural, rule-based and governance steps are
  visually distinct, because no neural score may override a rule gate.

Component choices are a reference bill of materials, not an endorsement;
each slot can be replaced by an equivalent (e.g. another NLI cross-encoder).
"""
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Patch  # noqa: E402

INK, INK2, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"
KIND = {  # fill, edge
    "neural": ("#dbe8f8", "#2a78d6"),
    "rule": ("#eeede8", "#8a8984"),
    "gov": ("#fbefd2", "#c98500"),
    "data": ("#e9e4f8", "#6250d6"),
    "edge": ("#dcf2e9", "#1baf7a"),
    "human": ("#fde4ec", "#d55181"),
}
LEGEND = [("neural", "Neural network (pretrained model)"), ("rule", "Deterministic rule (pgmem code)"),
          ("gov", "Governance / permission check"), ("data", "Storage / connector"),
          ("edge", "On-device interaction"), ("human", "Person / trusted person")]


def _box(ax, x, y, w, h, title, body="", kind="rule", fs=8.6):
    fc, ec = KIND[kind]
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.25,rounding_size=0.9",
                                fc=fc, ec=ec, lw=1.6))
    ax.text(x + w / 2, y + h - 0.9, title, ha="center", va="top", fontsize=fs, fontweight="bold", color=INK)
    if body:
        ax.text(x + w / 2, y + h - 2.6, body, ha="center", va="top", fontsize=fs - 1.4, color=INK2, linespacing=1.3)
    return (x, y, w, h)


def _arrow(ax, p, q, color=INK2, style="-|>", lw=1.3, ls="-", text=None, rad=0.0):
    ax.add_patch(FancyArrowPatch(p, q, arrowstyle=style, mutation_scale=11, color=color, lw=lw, ls=ls,
                                 connectionstyle=f"arc3,rad={rad}", shrinkA=2, shrinkB=2))
    if text:
        ax.text((p[0] + q[0]) / 2 + 0.6, (p[1] + q[1]) / 2, text, fontsize=6.8, color=color, va="center",
                style="italic")


def _band(ax, y, h, label):
    ax.add_patch(FancyBboxPatch((1, y), 72.5, h, boxstyle="round,pad=0.2,rounding_size=1.2", fc="none",
                                ec="#d6d5cf", lw=1, ls=(0, (4, 3))))
    ax.text(-0.6, y + h / 2, label, fontsize=8.5, color=INK2, va="center", ha="center", rotation=90,
            fontweight="bold")


def system_architecture(out: Path | None = None, name: str = "arch_system.png"):
    fig, ax = plt.subplots(figsize=(17, 11.5))
    ax.set(xlim=(-1.5, 100), ylim=(0, 70))
    ax.axis("off")
    fig.patch.set_facecolor(SURFACE)
    ax.text(1, 69, "PGMem — neural-network realisation on real components", fontsize=15, fontweight="bold")
    ax.text(1, 66.9, "Logical roles of Figure 1 bound to concrete models, services and stores. "
                     "Every outgoing action passes the governance column at execution time (I1).",
            fontsize=9.5, color=INK2)

    # people
    _box(ax, 3, 58.5, 17, 5.5, "Person with memory impairment", "voice · touch · glanceable cards", "human")
    _box(ax, 55, 58.5, 18, 5.5, "Trusted person (L4 only)", "agreed scopes, e.g. missed\nsafety-critical tasks", "human")

    # interaction / edge
    _band(ax, 46, 11, "1 · INTERACTION (device)")
    _box(ax, 3, 47, 13.5, 8, "Speech in", "Silero VAD (1.8 M)\n→ Whisper large-v3-turbo\n809 M, uk/en, int8", "neural")
    _box(ax, 18, 47, 13.5, 8, "Accessible app", "Flutter UI, large targets,\nresumable tasks,\nnever reads delay as 'no'", "edge")
    _box(ax, 33, 47, 13.5, 8, "Speech out", "Piper TTS uk_UA / en_GB\n(VITS, ~20 M)\nor Coqui XTTS-v2", "neural")
    _box(ax, 48, 47, 12.5, 8, "Notifications", "FCM / APNs\nlocal alarms\n(L1–L3 prompts)", "edge")
    _box(ax, 62, 47, 11, 8, "Trusted channel", "Signal / SMS\n(Twilio API)\nqueued only", "edge")

    # agentic
    _band(ax, 21.5, 23.5, "2 · AGENTIC (local inference server)")
    _box(ax, 3, 35.5, 70, 8, "Coordinator — LangGraph state machine (local server, 8–16 GB GPU or Apple silicon)",
         "goal → plan → permission check → execute → observe → adapt (Fig. 2a) · episode states "
         "pending/active/completed/postponed/declined/unknown (I3)\nconflict resolution between prompts · "
         "idempotency keys for replay (I4)", "rule", fs=9.2)
    _box(ax, 3, 23, 13, 11, "Interaction role", "Llama-3.1-8B-Instruct\nQ4_K_M via llama.cpp\n/ vLLM; plain-language\n"
                                              "dialogue, comprehension\nchecks", "neural")
    _box(ax, 17.5, 23, 13, 11, "Memory role", "BGE-M3 encoder (568 M,\n1024-d dense + sparse)\n→ BGE-reranker-v2-m3\n"
                                             "(cross-encoder, 568 M)", "neural")
    _box(ax, 32, 23, 13, 11, "Verification role", "claim decomposer (LLM,\nJSON schema) → mDeBERTa\n-v3-base-XNLI (278 M)\n"
                                                  "ent(a_j, r) → S(a_j) (Eq. 4)", "neural")
    _box(ax, 46.5, 23, 13, 11, "Assistance role", "pgmem.assistance\nEq. 2: L_{k+1} gate G_k\nEq. 3: start-level\n"
                                                  "adaptation (m, L_min)", "rule")
    _box(ax, 61, 23, 12, 11, "Release & monitor", "pgmem.recall.release\nθ_low / θ_high (Eq. 5)\nTSR, FAR, F_ext, C\n"
                                                    "(Eqs. 6–7)", "rule")

    # data
    _band(ax, 2, 18, "3 · DATA")
    _box(ax, 3, 9.5, 24, 8.5, "Personal event base", "PostgreSQL 16 + pgvector\nr_i = (c, t, s, κ, σ, v, e) · versions "
                                                     "·\nPROV-O lineage · retention covers\nembeddings, caches, backups",
         "data")
    _box(ax, 29, 9.5, 21, 8.5, "Ingestion", "MQTT (Mosquitto) → normaliser\n→ evidence typing:\nplan / self_report / "
                                            "observation /\ninference / disputed", "rule")
    _box(ax, 52, 9.5, 21, 8.5, "Connectors (opt-in)", "CalDAV / Google Calendar API\nAndroid CallLog · Health Connect\n"
                                                      "/ HealthKit · BLE smart pill box\nvia Home Assistant", "data")
    _box(ax, 3, 3, 47, 5, "Audit log — append-only, hash-chained", "every prompt, action, record change and "
                                                                   "deletion event (content never retained)", "data")

    # governance column
    ax.add_patch(FancyBboxPatch((76, 3), 22, 61, boxstyle="round,pad=0.3,rounding_size=1.2",
                                fc=KIND["gov"][0], ec=KIND["gov"][1], lw=1.8))
    ax.text(87, 63, "GOVERNANCE", ha="center", va="top", fontsize=11, fontweight="bold")
    items = [("Policy engine", "Open Policy Agent (Rego)\n⇄ pgmem.PolicyEngine\nscopes = (recipient,\ncategory, action)"),
             ("Consent manager", "revisable scopes · periodic\nreminders of what is shared\nsupported decision-making"),
             ("Dispatch-time check (I1)", "ActionQueue.dispatch():\nre-evaluate permission\nimmediately before send"),
             ("Invalidation (I4)", "on correct / dispute / delete:\ndrop cached answers,\nsummaries, queued actions"),
             ("Escalation rules", "L4 only under agreed rule\n+ current permission"),
             ("Data minimisation", "on-device by default;\nno raw audio leaves device;\nGDPR Art. 9 · EU AI Act")]
    y = 59.5
    for t, b in items:
        _box(ax, 77.5, y - 8.2, 19, 8, t, b, "gov", fs=8.4)
        y -= 9.1

    # arrows
    _arrow(ax, (11, 58.5), (10, 55.3))
    _arrow(ax, (12, 58.5), (24, 55.3))
    _arrow(ax, (39.5, 47), (39.5, 43.8), style="<|-")
    _arrow(ax, (10, 47), (10, 43.8))
    _arrow(ax, (25, 47), (25, 43.8), style="<|-|>")
    _arrow(ax, (54, 47), (54, 43.8), style="<|-")
    _arrow(ax, (67, 47), (67, 43.8), style="<|-", color="#c98500", text="after I1 check")
    _arrow(ax, (67, 55), (64, 58.5), color="#d55181")
    _arrow(ax, (38, 35.5), (38, 34.2), style="<|-|>")
    for x in (9.5, 24, 52.5, 67):
        _arrow(ax, (x, 35.5), (x, 34.2), style="<|-|>")
    _arrow(ax, (24, 23), (15, 18.2), style="<|-", text="top-k, permission-filtered")
    _arrow(ax, (38.5, 23), (20, 18.2), style="<|-", text="record versions")
    _arrow(ax, (52, 13.75), (50, 13.75), style="-|>")
    _arrow(ax, (29, 13.75), (27, 13.75), style="-|>")
    for yy in (39.5, 28.5, 13.5):
        _arrow(ax, (76, yy), (73.6, yy), color="#c98500", style="<|-|>", lw=1.6)
    _arrow(ax, (76, 5.5), (50.3, 5.5), color="#c98500", style="-|>", text="audit events")

    ax.legend(handles=[Patch(fc=KIND[k][0], ec=KIND[k][1], label=l) for k, l in LEGEND], loc="lower center",
              bbox_to_anchor=(0.5, -0.04), ncol=6, fontsize=8.5, frameon=False)
    fig.tight_layout()
    if out is not None:
        fig.savefig(Path(out) / name, bbox_inches="tight", facecolor=SURFACE, dpi=170)
    return fig


def neural_recall_pipeline(out: Path | None = None, name: str = "arch_recall_pipeline.png"):
    fig, ax = plt.subplots(figsize=(17, 8.0))
    ax.set(xlim=(0, 100), ylim=(4, 52))
    ax.axis("off")
    fig.patch.set_facecolor(SURFACE)
    ax.text(1, 51, "Grounded recall as a neural pipeline (Section 4.4, Eqs. 4–5)", fontsize=15, fontweight="bold")
    ax.text(1, 48.8, "Neural models propose; rule gates dispose. A high neural score can never override a failed "
                     "permission, conflict, identity or freshness gate.", fontsize=9.5, color=INK2)

    w, h, y1, y2, y3 = 13.2, 11, 33, 17, 5
    top = [
        ("1. Audio → text", "16 kHz PCM → log-Mel\n128×3000 → Whisper enc.\n(32 layers, d=1280)\n→ dec. → tokens",
         "neural"),
        ("2. Query intent", "zero-shot NLI (mDeBERTa)\nlabels: scheduled /\noccurred / content\n→ claim kind of q",
         "neural"),
        ("3. Permission filter", "SQL WHERE scope ∋\nrequester AND category\nallowed (I1) — before\nretrieval", "gov"),
        ("4. Dense retrieval", "e5 / BGE-M3 encoder\nq, r → 384–1024-d\nL2-normalised\ncosine top-k (pgvector)",
         "neural"),
        ("5. Rerank", "cross-encoder\n[CLS] q [SEP] r\n→ relevance logit\nkeep k' ≤ 5", "neural"),
        ("6. Rule gates", "∅ → abstain\ndisputed → clarify\n>1 event identity → clarify\nsuperseded v → skip (I4)",
         "rule"),
    ]
    bottom = [
        ("7. Draft (LLM)", "Llama-3.1-8B-Instruct\nprompt = typed records\n+ evidence status;\nno free recollection",
         "neural"),
        ("8. Claim split", "atomic claims a_1..a_m\n(JSON schema:\nsubject, kind, value,\nasserted_status)", "neural"),
        ("9. Evidence-type check", "SUPPORTS[status]:\nplan ⊬ attended,\nself-report ⊬ observed\n(I2)", "rule"),
        ("10. NLI support", "mDeBERTa-v3-base-XNLI\n(278 M) softmax over\n{entail, neutral, contra}\nS(a_j)=max ent·κ_r (Eq. 4)",
         "neural"),
        ("11. Release rule", "S_min ≥ θ_high: answer+source\nθ_low ≤ S_min: uncertain\n< θ_low: abstain (Eq. 5)\n"
                           "θ calibrated on dev split", "rule"),
        ("12. Re-check & speak", "permission re-checked\nat release (I1) → TTS\n(Piper / XTTS) + source\n"
                               "date, evidence type", "gov"),
    ]
    xs = [1 + i * (w + 3.3) for i in range(6)]
    for x, (t, b, k) in zip(xs, top):
        _box(ax, x, y1, w, h, t, b, k, fs=9)
    for x, (t, b, k) in zip(xs[::-1], bottom):
        _box(ax, x, y2, w, h, t, b, k, fs=9)
    for i in range(5):
        _arrow(ax, (xs[i] + w + 0.3, y1 + h / 2), (xs[i + 1] - 0.3, y1 + h / 2), lw=1.6)
        _arrow(ax, (xs[i + 1] - 0.3, y2 + h / 2), (xs[i] + w + 0.3, y2 + h / 2), lw=1.6)
    _arrow(ax, (xs[5] + w / 2, y1 - 0.3), (xs[5] + w / 2, y2 + h + 0.3), lw=1.6, text="records pass")
    _arrow(ax, (xs[5] + w + 0.4, y1 + 3), (99, 31), color="#e34948", text="")
    ax.text(97.5, 30, "clarify /\nabstain", fontsize=8, color="#e34948", ha="center", va="top")

    # outcome strip: what is NOT neural
    notes = [("Training / calibration", "NLI & reranker used zero-shot; θ_low, θ_high fitted on development histories,\n"
                                         "reported on held-out ones (pgmem.stats.calibrate_threshold)."),
             ("Failure modes kept visible", "retrieval of the wrong event (wrong_event), noisy entailment,\n"
                                             "ASR errors — measured by F_ext and coverage, never hidden."),
             ("Runs in this notebook", "Steps 2 (zero-shot intent), 4 (e5-small) and 10 (mDeBERTa-XNLI) run live\n"
                                       "in pgmem.neural (with an offline lexical fallback).")]
    for i, (t, b) in enumerate(notes):
        x = 1 + i * 33.2
        _box(ax, x, y3, 31.8, 7.5, t, b, "rule" if i < 2 else "neural", fs=9)

    ax.legend(handles=[Patch(fc=KIND[k][0], ec=KIND[k][1], label=l) for k, l in LEGEND[:3]], loc="upper right",
              bbox_to_anchor=(1.0, 0.985), ncol=3, fontsize=8.5, frameon=False)
    fig.tight_layout()
    if out is not None:
        fig.savefig(Path(out) / name, bbox_inches="tight", facecolor=SURFACE, dpi=170)
    return fig


In [ ]:
%%writefile pgmem_project/pgmem/neural.py
"""Real neural components for the grounded-recall pipeline.

* ``NLIScorer``      – multilingual NLI cross-encoder that estimates
  ent(a_j, r) in Equation (4) (default: mDeBERTa-v3-base-XNLI, 278 M).
* ``DenseRetriever`` – sentence-embedding retriever (default:
  multilingual-e5-small, 118 M, 384-d) over permission-filtered records.
* ``IntentClassifier`` – zero-shot claim-kind detection with the same NLI model.
* ``NeuralRecall``   – free-text question -> intent -> retrieval ->
  ``GroundedRecall`` gates -> NLI support -> release rule.

Every class degrades to a transparent lexical fallback when ``transformers``
/ ``sentence-transformers`` or the model weights are unavailable (offline
runs, CI), and reports which backend it used. The rule gates and the
evidence-type check stay deterministic in both cases: a neural score is an
input to Eq. (4), never a substitute for I1–I4.
"""
from __future__ import annotations

import re
import warnings
from dataclasses import dataclass
from datetime import datetime
from typing import Callable, Sequence

import numpy as np

from .model import SUPPORTS, Record
from .recall import Answer, Claim, GroundedRecall, Question

NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
EMB_MODEL = "intfloat/multilingual-e5-small"

_TOKEN = re.compile(r"[\w\-]+", re.UNICODE)
_STOP = {"the", "a", "an", "of", "to", "on", "in", "at", "for", "was", "is", "my", "i", "did", "do", "me",
         "that", "this", "with", "by", "and", "or", "it", "have", "has", "been", "be"}


def _tokens(s: str) -> set[str]:
    return {t for t in _TOKEN.findall(s.lower()) if t not in _STOP}


def _device():
    try:
        import torch
        return "cuda" if torch.cuda.is_available() else "cpu"
    except ImportError:
        return "cpu"


# ------------------------------------------------------------------- NLI
class NLIScorer:
    """P(entailment | premise, hypothesis) from an NLI cross-encoder."""

    def __init__(self, model_name: str = NLI_MODEL, use_model: bool = True, batch_size: int = 16):
        self.model_name, self.batch_size = model_name, batch_size
        self.backend = "lexical-fallback"
        self._cache: dict[tuple[str, str], float] = {}
        if not use_model:
            return
        try:
            import torch
            from transformers import AutoModelForSequenceClassification, AutoTokenizer
            self._torch = torch
            self.tok = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(_device()).eval()
            labels = {v.lower(): int(k) for k, v in self.model.config.id2label.items()}
            self.entail_id = next(i for name, i in labels.items() if name.startswith("entail"))
            self.backend = model_name
        except Exception as exc:  # offline, no weights, no torch ...
            warnings.warn(f"NLI model unavailable ({type(exc).__name__}: {exc}); using lexical fallback")

    def entail(self, premises: Sequence[str], hypotheses: Sequence[str]) -> np.ndarray:
        pairs = list(zip(premises, hypotheses))
        todo = [p for p in dict.fromkeys(pairs) if p not in self._cache]
        if todo:
            scores = self._model_scores(todo) if self.backend != "lexical-fallback" else \
                [self._lexical(p, h) for p, h in todo]
            self._cache.update(zip(todo, map(float, scores)))
        return np.array([self._cache[p] for p in pairs])

    def _model_scores(self, pairs):
        out = []
        for i in range(0, len(pairs), self.batch_size):
            chunk = pairs[i:i + self.batch_size]
            enc = self.tok([p for p, _ in chunk], [h for _, h in chunk], truncation=True, padding=True,
                           return_tensors="pt").to(self.model.device)
            with self._torch.no_grad():
                probs = self.model(**enc).logits.softmax(-1)[:, self.entail_id]
            out.extend(probs.cpu().numpy().tolist())
        return out

    @staticmethod
    def _lexical(premise: str, hypothesis: str) -> float:
        """Share of hypothesis content words found in the premise; a crude,
        clearly-labelled stand-in, so offline runs stay executable."""
        h, p = _tokens(hypothesis), _tokens(premise)
        return 0.0 if not h else 0.95 * len(h & p) / len(h)

    def matrix(self, premises: Sequence[str], hypotheses: Sequence[str]):
        import pandas as pd
        P = [p for p in premises for _ in hypotheses]
        H = [h for _ in premises for h in hypotheses]
        return pd.DataFrame(self.entail(P, H).reshape(len(premises), len(hypotheses)),
                            index=list(premises), columns=list(hypotheses))


# ----------------------------------------------------- verbalisation (I2)
def humanise_subject(subject: str) -> str:
    kind, *rest = subject.split(":")
    names = {"appt": "the appointment", "med": "the morning medication", "call": "the call",
             "bill": "the bill", "note": "the note"}
    rest = [r for r in rest if not r.startswith("P") or not r[1:].isdigit()]
    return " ".join([names.get(kind, kind)] + [r.replace("_", " ") for r in rest])


def verbalise_record(r: Record) -> str:
    when = r.time.strftime("%d %B %Y at %H:%M")
    lead = {"plan": f"The calendar lists: {r.content}, planned for {when}.",
            "self_report": f"The person reported on {when}: {r.content}.",
            "observation": f"The source '{r.source}' observed on {when}: {r.content}.",
            "inference": f"The assistant inferred: {r.content}.",
            "disputed": f"A disputed entry says: {r.content}."}
    return lead.get(r.status.value, r.content)


def _date(v: str) -> str:
    try:
        return datetime.fromisoformat(v).strftime("%d %B %Y")
    except ValueError:
        return v


def verbalise_claim(c: Claim, label: str | None = None) -> str:
    what = label or humanise_subject(c.subject)
    if c.kind == "scheduled":
        return f"{what} was planned for {_date(c.value)}."
    if c.kind == "occurred":
        verb = "took place" if c.value == "true" else "did not take place"
        if c.asserted_status.value == "self_report":
            return f"The person reported that {what} {verb}."
        return f"{what} {verb}."
    return f"The note says: {c.value}."


def neural_sup(scorer: NLIScorer) -> Callable[[Claim, Record], float]:
    """ent(a_j, r) for Eq. (4). The evidence-type check (I2) is applied first
    and is not learnable: a plan never supports 'it happened', however
    fluent the entailment model finds the pair."""
    def f(claim: Claim, r: Record) -> float:
        if r.status not in SUPPORTS.get(claim.asserted_status, set()):
            return 0.0
        return float(scorer.entail([verbalise_record(r)], [verbalise_claim(claim, r.content)])[0])
    return f


# -------------------------------------------------------------- retrieval
class DenseRetriever:
    """Cosine top-k over e5 embeddings ("query: " / "passage: " prefixes)."""

    def __init__(self, model_name: str = EMB_MODEL, use_model: bool = True):
        self.model_name, self.backend = model_name, "lexical-fallback"
        if use_model:
            try:
                from sentence_transformers import SentenceTransformer
                self.model = SentenceTransformer(model_name, device=_device())
                self.backend = model_name
            except Exception as exc:
                warnings.warn(f"Embedding model unavailable ({type(exc).__name__}: {exc}); using lexical fallback")

    def encode(self, texts: Sequence[str], prefix: str) -> np.ndarray:
        if self.backend != "lexical-fallback":
            return np.asarray(self.model.encode([prefix + t for t in texts], normalize_embeddings=True))
        dim = 512
        out = np.zeros((len(texts), dim))
        for i, t in enumerate(texts):
            for tok in _tokens(t):
                out[i, hash(tok) % dim] += 1.0
        return out / np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-9)

    def scores(self, query: str, records: Sequence[Record]) -> np.ndarray:
        if not records:
            return np.zeros(0)
        q = self.encode([query], "query: ")
        d = self.encode([f"{verbalise_record(r)} ({humanise_subject(r.subject)})" for r in records], "passage: ")
        return (d @ q.T).ravel()


# ------------------------------------------------------- intent detection
INTENT_LABELS = {
    "scheduled": "This question asks when something is planned or scheduled.",
    "occurred": "This question asks whether something actually happened.",
    "content": "This question asks what a written note says.",
}


class IntentClassifier:
    def __init__(self, scorer: NLIScorer):
        self.scorer = scorer

    def __call__(self, question: str) -> tuple[str, dict[str, float]]:
        if self.scorer.backend == "lexical-fallback":
            ql = question.lower()
            kind = "content" if "note" in ql else "scheduled" if re.search(r"\bwhen\b", ql) else "occurred"
            return kind, {k: float(k == kind) for k in INTENT_LABELS}
        s = self.scorer.entail([question] * len(INTENT_LABELS), list(INTENT_LABELS.values()))
        probs = {k: float(v) for k, v in zip(INTENT_LABELS, (s / s.sum()).round(3))}
        return max(probs, key=probs.get), probs


# ------------------------------------------------------- identity check
_NOT_NAMES = {"I", "Dr", "Did", "When", "What", "Who", "Was", "Is", "Has", "Have", "Do", "Does", "My", "The"}
_DATE = re.compile(r"\d{4}-\d{2}-\d{2}")


def identity_ok(query: str, r: Record) -> bool:
    """Deterministic event-identity check: every proper name and ISO date in
    the question must occur in the record. Dense similarity alone happily
    matches 'Did Andrii call?' to a call from Oleh (Table 3, row 1 logic)."""
    text = f"{r.subject} {r.content} {r.value} {r.time.date()}".lower()
    names = {w for w in re.findall(r"\b[A-Z][a-z]+\b", query) if w not in _NOT_NAMES}
    return all(n.lower() in text for n in names) and all(d in text for d in _DATE.findall(query))


# --------------------------------------------------------------- pipeline
@dataclass
class NeuralTrace:
    question: str
    intent: str
    intent_probs: dict
    retrieved: list[tuple[str, float]]
    answer: Answer


class NeuralRecall:
    """Free-text recall: neural steps propose, ``GroundedRecall`` disposes."""

    def __init__(self, recall: GroundedRecall, retriever: DenseRetriever, intent: IntentClassifier,
                 top_k: int = 3, min_similarity: float | None = None, margin: float = 0.03):
        self.recall, self.retriever, self.intent = recall, retriever, intent
        if min_similarity is None:   # e5 cosines are compressed towards 1; the hashed fallback is not
            min_similarity = 0.80 if retriever.backend != "lexical-fallback" else 0.20
        self.top_k, self.min_similarity, self.margin = top_k, min_similarity, margin
        self._query = ""
        self._last: list[tuple[str, float]] = []
        recall.retriever = self._retrieve

    def _retrieve(self, q: Question, permitted: list[Record]) -> list[Record]:
        # Only permission-filtered records reach the encoder (I1); records that
        # contradict a name or date in the question never reach the gates.
        permitted = [r for r in permitted if identity_ok(self._query, r)]
        sims = self.retriever.scores(self._query, permitted)
        order = np.argsort(-sims)[: self.top_k]
        self._last = [(permitted[i].subject, float(sims[i])) for i in order]
        if not len(order) or sims[order[0]] < self.min_similarity:
            return []
        best = sims[order[0]]
        # keep every record whose similarity is within the margin of the best:
        # two different events this close trigger the identity gate (clarify).
        near = {permitted[i].subject for i in order if best - sims[i] <= self.margin}
        return [r for r in permitted if r.subject in near]

    def ask(self, text: str, person_id: str, requester: str | None = None, category: str = "*") -> NeuralTrace:
        kind, probs = self.intent(text)
        self._query = text
        self.recall._cache.clear()
        q = Question("nq", person_id, requester or person_id, subject="?", kind=kind, category=category, text=text)
        ans = self._answer(q)
        return NeuralTrace(text, kind, probs, self._last, ans)

    def _answer(self, q: Question) -> Answer:
        recs = self.recall._retrieve(q)
        subjects = {r.subject for r in recs}
        if len(subjects) == 1:
            subject = subjects.pop()
            cat = next(r.category for r in recs)
            q = Question(q.question_id, q.person_id, q.requester, subject, q.kind, cat, q.text)
            return self.recall._answer(q)
        if not recs:
            return Answer("abstain", text="I don't have any record that matches this question, so I cannot know it.")
        return Answer("clarify", text="I found records about several similar events. Which one do you mean?")


In [ ]:
%%writefile pgmem_project/pgmem/pipeline.py
"""End-to-end pipeline: generate -> simulate -> analyse -> report."""
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from . import plots, schema, simulate, stats, synth

DEFAULTS = dict(n=240, days=28, seed=2026)


def generate_dataset(data_dir: Path, n: int = 240, days: int = 28, seed: int = 2026) -> dict[str, pd.DataFrame]:
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    cohort = synth.generate_cohort(n, seed)
    episodes = synth.generate_episodes(cohort, days)
    records, questions, reference = synth.generate_memory_corpus(cohort, seed + 1)
    tables = dict(participants=cohort, episodes=episodes, records=records, questions=questions,
                  reference_history=reference)
    for name, df in tables.items():
        df.to_csv(data_dir / f"{name}.csv", index=False)
    schema.write_descriptor(data_dir / "datapackage.json", seed, n, days)
    return tables


def load_dataset(data_dir: Path) -> dict[str, pd.DataFrame]:
    data_dir = Path(data_dir)
    t = {k: pd.read_csv(data_dir / f"{k}.csv") for k in schema.RESOURCES}
    t["records"]["supersedes"] = t["records"]["supersedes"].fillna("")
    return t


def run_simulation(t: dict[str, pd.DataFrame], results_dir: Path, seed: int = 2026):
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    ep = simulate.simulate_episodes(t["participants"], t["episodes"], seed + 10)
    rc = simulate.simulate_recall(t["participants"], t["records"], t["questions"], t["reference_history"], seed + 20)
    ep.to_csv(results_dir / "episode_outcomes.csv", index=False)
    rc.to_csv(results_dir / "recall_outcomes.csv", index=False)
    return ep, rc


HEADLINE = ["TSR", "FPR", "independence", "invented_outcome_rate", "F_ext", "coverage",
            "fabricated_per_100", "overstated_per_100", "appropriate_behaviour"]


def analyse(t, ep, rc, results_dir: Path, reports_dir: Path, n_boot: int = 300) -> dict[str, pd.DataFrame]:
    results_dir, reports_dir = Path(results_dir), Path(reports_dir)
    reports_dir.mkdir(parents=True, exist_ok=True)
    cohort = t["participants"]
    pm = stats.participant_metrics(ep, rc)
    out = {
        "cohort_table": stats.describe_cohort(cohort),
        "severity": stats.memory_impairment_profile(cohort),
        "participant_metrics": pm,
        "by_condition": pd.concat({m: stats.cluster_bootstrap(pm, m, n_boot) for m in HEADLINE}, axis=1),
        "by_condition_diagnosis": stats.pooled(pm.merge(cohort[["participant_id", "diagnosis"]]),
                                               ["diagnosis", "condition"]),
        "paired": stats.paired_differences(pm, ["TSR", "FPR", "independence", "appropriate_behaviour"]),
        "group_tests": stats.group_tests(pm, cohort),
        "logit": stats.logistic_completion(ep, cohort),
        "strata": stats.recall_by_stratum(rc),
        "safety": ep.groupby("condition")[["unauthorised_contact", "contact_cancelled", "wrong_failure",
                                           "missed_prompts"]].sum(),
    }
    for k, df in out.items():
        df.to_csv(results_dir / f"{k}.csv", index=k in ("cohort_table", "severity", "by_condition", "safety"))
    out["risk_coverage"] = stats.risk_coverage(rc)
    out["calibration"] = stats.calibrate_threshold(rc)
    out["risk_coverage"].to_csv(results_dir / "risk_coverage.csv", index=False)
    make_figures(out, t, ep, rc, reports_dir)
    write_report(out, t, reports_dir / "summary.md")
    return out


def make_figures(out, t, ep, rc, reports_dir: Path, interactive_html: bool = True) -> None:
    from . import architecture
    from .recall import THETA_HIGH, THETA_LOW
    cohort, pm = t["participants"], out["participant_metrics"]
    plots.cohort_overview(cohort, reports_dir)
    plots.condition_radar(out["by_condition"], reports_dir)
    plots.condition_dots(out["by_condition"], ["TSR", "FPR", "independence", "invented_outcome_rate"], reports_dir)
    plots.condition_dots(out["by_condition"], ["F_ext", "coverage", "fabricated_per_100", "appropriate_behaviour"],
                         reports_dir, "fig2c_dots_recall.png")
    plots.paired_forest(out["paired"], reports_dir)
    plots.tsr_raincloud(pm, cohort, reports_dir)
    plots.recall_heatmap(out["strata"], reports_dir)
    plots.invariant_scoreboard(ep, rc, reports_dir)
    plots.risk_coverage(out["risk_coverage"], THETA_LOW, THETA_HIGH, reports_dir, out["calibration"])
    plots.level_distribution(ep, reports_dir)
    plots.adaptation_over_time(ep, reports_dir)
    architecture.system_architecture(reports_dir)
    architecture.neural_recall_pipeline(reports_dir)
    plt.close("all")
    if interactive_html:
        try:
            from . import interactive as ix
        except ImportError:      # plotly is optional for the static report
            return
        html = Path(reports_dir) / "interactive"
        html.mkdir(exist_ok=True)
        ix.episode_sankey(ep, html)
        ix.recall_sunburst(rc, html)
        ix.condition_radar(out["by_condition"], html)
        ix.cohort_parallel(cohort, pm, html)
        ix.weekly_trajectories(ep, cohort, html)
        ix.risk_coverage(out["risk_coverage"], THETA_LOW, html)


def _md(df: pd.DataFrame, floatfmt: str = ".3f", index: bool = True) -> str:
    return df.to_markdown(floatfmt=floatfmt, index=index)


FIGURES = ["arch_system.png", "arch_recall_pipeline.png", "fig1_cohort.png", "fig2_radar.png", "fig2b_dots.png",
           "fig2c_dots_recall.png", "fig3_forest.png", "fig4_raincloud.png", "fig5_recall_heatmap.png",
           "fig6_invariants.png", "fig7_risk_coverage.png", "fig8_levels.png", "fig9_adaptation.png"]


def write_report(out, t, path: Path) -> None:
    bc = out["by_condition"].xs("estimate", axis=1, level=1)
    cal = out["calibration"]
    lines = [
        "# Report: provenance-governed memory assistant on synthetic data",
        "",
        "> All data are synthetic. Results illustrate how the framework rules behave, not effectiveness for real people.",
        "",
        f"Participants: {len(t['participants'])}, episodes: {len(t['episodes'])}, "
        f"event-base records: {len(t['records'])}, questions: {len(t['questions'])}.",
        "",
        "## 1. Cohort characteristics (metadata)",
        "", _md(out["cohort_table"]), "",
        "## 2. Memory-impairment severity by MoCA", "", _md(out["severity"]), "",
        "## 3. Indicators by condition (pooled; 95% cluster-bootstrap CIs in results/by_condition.csv)",
        "", _md(bc), "",
        "## 4. Paired differences (proposed framework − baselines)", "", _md(out["paired"], index=False), "",
        "## 5. Safety and invariants", "", _md(out["safety"]), "",
        "* `unauthorised_contact` — trusted-person contact after consent withdrawal (violates I1).",
        "* `wrong_failure` — task was done but recorded as missed (violates I3).", "",
        "## 6. Metadata and outcome (proposed framework)", "", _md(out["group_tests"], index=False), "",
        "### Logistic regression of episode completion", "", _md(out["logit"], index=False), "",
        "## 7. Recall by stratum", "", _md(out["strata"], index=False), "",
        "## 8. Release-rule calibration (Eq. 5)", "",
        f"Lowest θ_low reaching F_ext ≥ 0.99 on development participants: **{cal['theta_low']:.2f}**; "
        f"held-out coverage {cal['test']['coverage']:.3f}, held-out F_ext {cal['test']['F_ext']:.3f} "
        f"(default θ_low = 0.40). Full curve: results/risk_coverage.csv.", "",
        "## 9. Figures", "",
        *[f"* ![{n}]({n}) " for n in FIGURES],
        "* Interactive (open in a browser): `interactive/*.html` — Sankey of episode flow, recall sunburst, "
        "radar, parallel coordinates, weekly animation, risk–coverage.", "",
    ]
    path.write_text("\n".join(lines), encoding="utf-8")


In [ ]:
# (Optional) use the zip bundle instead of the embedded code:
USE_ZIP = False
if USE_ZIP:
    from google.colab import files
    up = files.upload()                      # choose pgmem_project.zip
    import zipfile
    zipfile.ZipFile(next(iter(up))).extractall(".")
    PROJECT = "memory_support_pkm"; sys.path.insert(0, PROJECT)

In [ ]:
import importlib, json, warnings
import pgmem
importlib.reload(pgmem)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
from pgmem import pipeline, schema, stats, plots, simulate, architecture
from pgmem import interactive as ix
pd.set_option("display.width", 200, "display.max_columns", 40)
warnings.filterwarnings("ignore", category=FutureWarning)
REPORTS = f"{PROJECT}/reports"; os.makedirs(REPORTS, exist_ok=True)
print("pgmem", pgmem.__version__)

## 1. Synthetic dataset and metadata

Parameters: number of participants, observation period (days) and seed for reproducibility.

In [ ]:
N_PARTICIPANTS, DAYS, SEED = 240, 28, 2026
DATA = f"{PROJECT}/data"
tables = pipeline.generate_dataset(DATA, N_PARTICIPANTS, DAYS, SEED)
for k, df in tables.items():
    print(f"{k:18s} {df.shape}")

### 1.1 Metadata (`datapackage.json`)

Every table is described by a schema: field type, description, range, allowed values. The same schema **validates** any dataset — including a real one collected after ethical approval.

In [ ]:
desc = json.load(open(f"{DATA}/datapackage.json", encoding="utf-8"))
print(desc["title"]); print(desc["description"]); print("generator:", desc["generator"])
meta = pd.DataFrame([dict(table=r["name"], field=f["name"], type=f["type"], description=f["description"],
                          constraints=json.dumps(f.get("constraints", {}), ensure_ascii=False))
                     for r in desc["resources"] for f in r["schema"]["fields"]])
display(meta[meta.table == "participants"])
print("Validation problems:", schema.validate(DATA) or "none")

In [ ]:
t = pipeline.load_dataset(DATA)
cohort = t["participants"]
display(cohort.head())
display(t["records"].head())
display(t["questions"].sample(5, random_state=1)[["stratum", "requester", "text", "ideal"]])

## 2. Statistics of people with memory impairment (metadata)

**Cohort fingerprint:** age ridgelines per diagnosis; the MoCA × PRMQ landscape with density contours (thick = 70 % of peak density); and a z-score heat-strip that shows at a glance how each diagnostic group departs from the cohort mean.

In [ ]:
display(Markdown("### Cohort characteristics by diagnosis"))
display(stats.describe_cohort(cohort))
display(Markdown("### Severity by MoCA × diagnosis"))
display(stats.memory_impairment_profile(cohort))

In [ ]:
plots.cohort_overview(cohort); plt.show()
num = ["age", "moca", "prmq_prospective", "prmq_retrospective", "education_years", "digital_literacy", "months_since_diagnosis"]
display(Markdown("### Spearman correlations between metadata"))
display(cohort[num].corr(method="spearman").round(2).style.background_gradient(cmap="RdBu", vmin=-1, vmax=1))

In [ ]:
display(Markdown("### Age bands and living situation"))
display(pd.crosstab(cohort.age_band, cohort.diagnosis, margins=True))
display(cohort.groupby("diagnosis")[["lives_alone", "has_trusted_person", "sensors_connected", "uses_voice"]].mean().round(2))

## 3. Neural-network solution on real components

The paper's Figure 1 names **logical roles**. The diagrams below bind every role to a concrete, openly available component so the design can be built, benchmarked and audited:

* **blue** — pretrained neural networks (ASR, LLM, encoders, NLI cross-encoder, TTS);
* **grey** — deterministic `pgmem` rules (Eqs. 2–5, evidence typing, gates);
* **amber** — governance: permission is checked at retrieval, at release and again at dispatch (I1).

The key design rule: *neural models propose, rule gates dispose* — a high entailment score can never override a failed permission, conflict, identity or freshness check.

In [ ]:
architecture.system_architecture(REPORTS); plt.show()

In [ ]:
architecture.neural_recall_pipeline(REPORTS); plt.show()

In [ ]:
bom = pd.DataFrame([
    ("Speech in", "Silero VAD → Whisper large-v3-turbo", "neural", "1.8 M + 809 M", "MIT", "device / server"),
    ("Speech out", "Piper TTS (uk_UA, en_GB) / Coqui XTTS-v2", "neural", "≈20 M / 467 M", "MIT / CPML", "device"),
    ("Interaction role", "Llama-3.1-8B-Instruct, Q4_K_M (llama.cpp / vLLM)", "neural", "8 B", "Llama 3.1 community", "local server"),
    ("Memory role: encoder", "BGE-M3 (prod.) / multilingual-e5-small (this notebook)", "neural", "568 M / 118 M", "MIT", "server"),
    ("Memory role: reranker", "BGE-reranker-v2-m3 cross-encoder", "neural", "568 M", "Apache-2.0", "server"),
    ("Verification: ent(a, r)", "mDeBERTa-v3-base-xnli-multilingual-nli-2mil7", "neural", "278 M", "MIT", "server"),
    ("Claim decomposition", "LLM with JSON-schema constrained decoding", "neural", "(shared 8 B)", "—", "server"),
    ("Assistance (Eqs. 2–3)", "pgmem.assistance", "rule", "—", "this work", "server"),
    ("Release rule (Eq. 5)", "pgmem.recall.release, θ calibrated (§8)", "rule", "—", "this work", "server"),
    ("Policy engine (I1)", "Open Policy Agent (Rego) ⇄ pgmem.PolicyEngine", "governance", "—", "Apache-2.0", "server"),
    ("Event base (Eq. 1)", "PostgreSQL 16 + pgvector, PROV-O lineage", "storage", "—", "PostgreSQL", "server"),
    ("Connectors", "CalDAV, Android CallLog, Health Connect / HealthKit, BLE pill box via MQTT", "storage", "—", "various", "device"),
], columns=["role", "component", "type", "parameters", "licence", "runs on"])
display(bom)

## 4. Paper scenarios on working code

Personal event base (Eq. 1), access policy, grounded recall (Eqs. 4–5).

In [ ]:
from datetime import datetime, timedelta
from pgmem import PersonalEventBase, PolicyEngine, GroundedRecall, Record, EvidenceStatus as ES, Question
from pgmem import ActionQueue, QueuedAction, next_level, gate

T = datetime(2026, 10, 2, 10)
base, pol = PersonalEventBase("olena"), PolicyEngine("olena")
gr = GroundedRecall(base, pol)
def add(subject, value, status, kind, cat, kappa=0.9, source="calendar", content=None):
    return base.add(Record(base.new_id(), "olena", subject, content or value, T, source, kappa, frozenset({"self"}),
                           status=status, category=cat, kind=kind, value=value))
def ask(subject, kind, cat, who="olena"):
    a = gr.answer(Question("q", "olena", who, subject, kind, cat))
    print(f"[{a.decision}] {a.text}")
    return a

print("Past appointment (only a plan exists):")
add("appt:Kovalenko", "2026-10-02", ES.PLAN, "scheduled", "appointments", 0.95, content="Appointment with Dr Kovalenko")
ask("appt:Kovalenko", "occurred", "appointments")
print("\nMedication (self-report at 08:47):")
add("med:2026-10-23", "true", ES.SELF_REPORT, "occurred", "medication", 0.85, "user_touch", "Morning medication taken")
ask("med:2026-10-23", "occurred", "medication")
print("\nAndrii's call (call log not connected):")
ask("call:Andrii:2026-10-22", "occurred", "social")
print("\nDisputed record (electricity bill):")
r1 = add("bill", "true", ES.SELF_REPORT, "occurred", "finance", 0.8, "user_voice", "Electricity bill paid")
r2 = add("bill", "false", ES.OBSERVATION, "occurred", "finance", 0.99, "carer_entry", "Bill not paid (carer entry)")
base.mark_disputed([r1.record_id, r2.record_id], "olena", T)
ask("bill", "occurred", "finance")
print("\nPersonal note, daughter asks without permission:")
add("note:1", "Wi-Fi password is on the fridge", ES.SELF_REPORT, "content", "personal_note", source="user_voice")
ask("note:1", "content", "personal_note", who="iryna");

In [ ]:
print("I1: consent withdrawn after a message was queued")
pol.grant("iryna", "medication", "message")
q = ActionQueue(pol)
q.enqueue(QueuedAction("msg-1", "message", "iryna", "medication", "Morning medication missed", T))
pol.revoke("iryna", "medication", "message")
print("sent:", q.dispatch(T + timedelta(minutes=5)), "| cancelled:", [a.key for a in q.cancelled])

print("\nI4: correcting a record invalidates the cached answer")
r = add("appt:dentist", "2026-10-10", ES.PLAN, "scheduled", "appointments", content="Dentist")
ask("appt:dentist", "scheduled", "appointments")
base.correct(r.record_id, "olena", T, value="2026-10-14", content="Dentist (rescheduled)")
ask("appt:dentist", "scheduled", "appointments")

print("\nEquation (2): escalation L1→L2 only when G_k = 1")
for postponed, progress in [(False, True), (True, True), (False, False)]:
    g = gate(True, postponed, False, progress)
    print(f"  postponed={postponed}, progress_check={progress}: G={g}, L→{next_level(1, g, 4)}")

print("\nAudit trail:"); display(pd.DataFrame([vars(e) for e in base.audit]).tail(6))

## 5. Comparative simulation of three conditions

The same synthetic people and days (common random numbers) for: **reminder app** (fixed L1, silence = missed), **single agent + RAG** (escalates on timeout, no dispatch-time consent check, no evidence types) and the **proposed framework**.

In [ ]:
%%time
ep, rc = pipeline.run_simulation(t, f"{PROJECT}/results", SEED)
out = pipeline.analyse(t, ep, rc, f"{PROJECT}/results", REPORTS, n_boot=300)
display(Markdown("### Pooled indicators (estimate and 95% cluster-bootstrap CI)"))
display(out["by_condition"].round(3))

**How to read:**
* **TSR** = N_c/N_s — share of completed tasks; **FPR** (FAR in the paper) = N_f/N_p — share of unnecessary prompts (task already done, or after postponement/decline).
* **independence** — share of completions at L0–L1. For the reminder app it is trivially 1.0 because no other levels exist.
* **invented_outcome_rate** — episodes where silence was turned into "missed" (violates I3).
* **F_ext** = N_e/N_r — share of released claims supported by the reference history; **coverage** = N_a/N_q. F_ext is undefined (NaN) when no claims are released.
* The baseline agent's higher TSR is achieved by escalating despite postponement and by contacts without valid consent — under proposition P3 this is **not** success.

### 5.1 Radar profile — every axis oriented so that the outer ring is better
Bands are 95 % cluster-bootstrap intervals; × marks an undefined indicator. The interactive version shows raw values on hover.

In [ ]:
plots.condition_radar(out["by_condition"]); plt.show()
ix.condition_radar(out["by_condition"]).show()

In [ ]:
plots.condition_dots(out["by_condition"], ["TSR", "FPR", "independence", "invented_outcome_rate"]); plt.show()
plots.condition_dots(out["by_condition"], ["F_ext", "coverage", "fabricated_per_100", "appropriate_behaviour"]); plt.show()

### 5.2 Where does silence become a fabricated outcome? (interactive Sankey)
Condition → true outcome → recorded state. **Red ribbons** are tasks that were in fact completed but recorded as missed — a direct violation of I3. In the proposed framework these flows end in *unknown*, *postponed* or *declined* instead.

In [ ]:
ix.episode_sankey(ep).show()

### 5.3 Invariant scoreboard and paired differences

In [ ]:
plots.invariant_scoreboard(ep, rc); plt.show()
plots.paired_forest(out["paired"]); plt.show()
display(Markdown("### Safety and invariants")); display(out["safety"])
display(Markdown("### Paired differences per participant")); display(out["paired"].round(4))

### 5.4 Assistance levels and adaptation (Eqs. 2–3)
Diverging bars put *independent* completions (L0–L1) to the right and *more help* (L2–L4) to the left. The adaptation panel shows the starting level falling over the 28 days for routine tasks — with the user's agreement only — while the safety-critical medication task never adapts.

In [ ]:
plots.level_distribution(ep); plt.show()
plots.adaptation_over_time(ep); plt.show()

### 5.5 Week-by-week animation
Each bubble is one synthetic participant: x = MoCA, y = cumulative task success, size = escalations so far. Press ▶ to watch the conditions separate.

In [ ]:
ix.weekly_trajectories(ep, cohort).show()

## 6. Metadata × outcomes: who benefits least?

In [ ]:
display(out["by_condition_diagnosis"][["diagnosis", "condition", "TSR", "FPR", "independence", "F_ext", "coverage"]].round(3))
plots.tsr_raincloud(out["participant_metrics"], cohort); plt.show()

**Parallel coordinates** — every line is one person, coloured by TSR under the proposed framework. *Drag along any axis* to brush a subgroup (e.g. MoCA ≤ 17 or digital literacy = 1) and see where their outcomes land.

In [ ]:
ix.cohort_parallel(cohort, out["participant_metrics"]).show()

In [ ]:
display(Markdown("### Statistical tests (proposed framework)")); display(out["group_tests"].round(4))
display(Markdown("### Logistic regression of episode completion (OR, 95% CI)")); display(out["logit"].round(3))
pm = out["participant_metrics"].merge(cohort, on="participant_id")
p = pm[pm.condition == "proposed"]
p = p.assign(moca_band=pd.cut(p.moca, [0, 17, 21, 25, 30], labels=["≤17", "18–21", "22–25", "26–30"]))
display(p.groupby("moca_band", observed=True)[["TSR", "independence", "FPR", "coverage"]].agg(["mean", "count"]).round(3))

## 7. Recall by stratum (Table 3 counterexamples)
The heatmap colour is the share of reference behaviour; the second line in each cell lists the invariant violations that a plain accuracy score would hide (e.g. the baseline is "100 % appropriate" on attendance questions while overstating 720 plans as attendance).

In [ ]:
plots.recall_heatmap(out["strata"]); plt.show()
ix.recall_sunburst(rc).show()

In [ ]:
display(out["strata"].round(3))
display(Markdown("### Example answers"))
display(rc[rc.stratum.isin(["attendance", "unanswerable", "contradictory", "consent_restricted"])]
        .groupby(["stratum", "condition"]).head(1)[["stratum", "condition", "decision", "answer_text"]])

## 8. Calibrating the release rule (Eq. 5)
The paper requires θ_low and θ_high to be *calibrated on development data and tested on held-out cases*. The risk–coverage curve shows what each θ_low trades: lower thresholds answer more questions (coverage C) at the cost of faithfulness (F_ext). Participants are split 50/50; the lowest θ_low reaching F_ext ≥ 0.99 on the development half is reported on the held-out half (★).

In [ ]:
from pgmem.recall import THETA_LOW, THETA_HIGH
cal = out["calibration"]
print(f"calibrated θ_low = {cal['theta_low']:.2f}")
display(pd.DataFrame({"development": cal["dev"], "held-out": cal["test"]}).T.round(4))
plots.risk_coverage(out["risk_coverage"], THETA_LOW, THETA_HIGH, calibrated=cal); plt.show()
ix.risk_coverage(out["risk_coverage"], THETA_LOW).show()

## 9. Live neural components

This section runs real pretrained networks in place of the simulation's noisy scorer:

| Step | Model | Role in the paper |
|---|---|---|
| NLI entailment | `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` (278 M, 100+ languages incl. Ukrainian) | ent(a_j, r) in Eq. (4) |
| Zero-shot intent | same NLI model | which claim kind the question asks for |
| Dense retrieval | `intfloat/multilingual-e5-small` (118 M, 384-d) | Memory role, R(q) |

If the weights cannot be downloaded (no internet), each component falls back to a transparent lexical scorer and says so — the notebook still runs end to end. Evidence typing (I2), permission (I1), identity and conflict gates remain deterministic either way.

In [ ]:
RUN_NEURAL = True          # set False to skip downloads
%pip -q install transformers sentence-transformers sentencepiece
from pgmem import neural
nli = neural.NLIScorer(use_model=RUN_NEURAL)
emb = neural.DenseRetriever(use_model=RUN_NEURAL)
print("NLI backend:      ", nli.backend)
print("Embedding backend:", emb.backend)

### 9.1 Why a plan does not entail attendance
Rows are records (premises) written the way the event base types them; columns are claims an assistant might make. A well-behaved NLI model gives a high score where the record really supports the claim and a low score for *"attended"* given only a calendar entry — the gap that I2 hard-codes.

In [ ]:
premises = [
    "The calendar lists: Appointment with Dr Kovalenko, planned for 02 October 2026 at 10:00.",
    "The person reported on 23 October 2026 at 08:47: Morning medication taken.",
    "The source 'call_log' observed on 22 October 2026 at 19:00: Incoming call: Oleh.",
    "The person reported on 20 October 2026 at 20:00: Wi-Fi password is on the fridge.",
]
hypotheses = [
    "The appointment with Dr Kovalenko was planned for 02 October 2026.",
    "Olena attended the appointment with Dr Kovalenko.",
    "The person reported taking the morning medication.",
    "A sensor confirmed the medication was taken.",
    "Oleh called.",
    "Andrii called.",
    "The note says the Wi-Fi password is on the fridge.",
]
mat = nli.matrix(premises, hypotheses)
plots.entailment_heatmap(mat, f"P(entailment) — {nli.backend.split('/')[-1]}"); plt.show()

### 9.2 End-to-end neural recall on Olena's event base
Free-text question → zero-shot intent → e5 retrieval over **permission-filtered** records → deterministic identity check (names and dates in the question must match the record) → `GroundedRecall` gates → NLI support score → release rule.

In [ ]:
from pgmem.neural import NeuralRecall, IntentClassifier, neural_sup
T2 = datetime(2026, 10, 22, 19)
ob, op = PersonalEventBase("olena"), PolicyEngine("olena")
op.grant("trusted:olena", "medication", "read")
def put(subject, content, value, status, kind, cat, kappa=0.9, source="calendar", when=T):
    ob.add(Record(ob.new_id(), "olena", subject, content, when, source, kappa, frozenset({"self"}),
                  status=status, category=cat, kind=kind, value=value))
put("appt:Kovalenko:2026-10-02", "Appointment with Dr Kovalenko", "2026-10-02", ES.PLAN, "scheduled", "appointments", 0.95)
put("med:morning:2026-10-23", "Morning medication taken", "true", ES.SELF_REPORT, "occurred", "medication", 0.85, "user_touch")
put("call:Oleh:2026-10-22", "Incoming call: Oleh", "true", ES.OBSERVATION, "occurred", "social", 0.98, "call_log", T2)
put("note:olena:0", "Wi-Fi password is on the fridge", "Wi-Fi password is on the fridge", ES.SELF_REPORT, "content", "personal_note", 0.9, "user_voice")

nr = NeuralRecall(GroundedRecall(ob, op, sup=neural_sup(nli)), emb, IntentClassifier(nli))
rows = []
for text, who in [("When is my appointment with Dr Kovalenko?", "olena"),
                  ("Did I actually go to the appointment with Dr Kovalenko?", "olena"),
                  ("Did I take my morning medication?", "olena"),
                  ("Did Oleh call me?", "olena"),
                  ("Did Andrii call me yesterday?", "olena"),
                  ("What does my note about the Wi-Fi say?", "olena"),
                  ("What does her note about the Wi-Fi say?", "trusted:olena")]:
    tr = nr.ask(text, "olena", requester=who)
    rows.append(dict(asked_by=who, question=text, intent=tr.intent,
                     top_hit=f"{tr.retrieved[0][0]} ({tr.retrieved[0][1]:.2f})" if tr.retrieved else "—",
                     decision=tr.answer.decision, S_min=None if tr.answer.s_min is None else round(tr.answer.s_min, 3),
                     answer=tr.answer.text))
display(pd.DataFrame(rows))

### 9.3 Swapping the simulated scorer for the NLI network
The rule-level comparison of §5 used a *noisy reference scorer*. Here the proposed condition's ent(a, r) is computed by the NLI network for a subset of participants, and the release behaviour is compared on identical questions. Expect lower coverage if the network is conservative — that is the honest cost of real entailment, reported alongside F_ext as the paper requires.

In [ ]:
N_NEURAL = 30   # participants (raise on GPU)
sub = cohort.head(N_NEURAL)
rc_noisy = simulate.simulate_recall(sub, t["records"], t["questions"], t["reference_history"], SEED + 20)
rc_nli = simulate.simulate_recall(sub, t["records"], t["questions"], t["reference_history"], SEED + 20,
                                  sup_factory=lambda rng: neural_sup(nli))
def summary(r, label):
    p = r[r.condition == "proposed"]
    rel = p.claims_released.sum()
    return dict(scorer=label, questions=len(p), coverage=p.substantive.mean(),
                F_ext=p.claims_supported.sum() / rel if rel else float("nan"),
                appropriate=p.appropriate.mean(), uncertain_answers=(p.decision == "answer_uncertain").mean(),
                overstated=int(p.overstated.sum()), consent_violations=int(p.consent_violation.sum()))
display(pd.DataFrame([summary(rc_noisy, "simulated noisy scorer"), summary(rc_nli, f"NLI: {nli.backend}")]).round(3))
fig, ax = plt.subplots(figsize=(8, 3))
for r, lbl, c in [(rc_noisy, "simulated", "#8a8984"), (rc_nli, "NLI network", "#2a78d6")]:
    s = r[(r.condition == "proposed") & r.s_min.notna()].s_min
    ax.hist(s, bins=30, range=(0, 1), alpha=0.6, color=c, label=lbl)
for th in (THETA_LOW, THETA_HIGH):
    ax.axvline(th, color="#0b0b0b", ls="--", lw=1)
ax.set(xlabel="support score S_min", ylabel="answers", title="Support-score distribution vs release thresholds")
ax.legend(); plt.show()

## 10. Your own data

To analyse a real (ethically approved, pseudonymised) dataset: put CSVs with the same names and columns into a folder together with `datapackage.json`, run `schema.validate(...)`, then `pipeline.load_dataset` → `run_simulation`/`analyse`. The `ability_latent` column is needed only by the simulation.

In [ ]:
# MY_DATA = "my_data"          # folder with participants.csv, episodes.csv, ... and datapackage.json
# print(schema.validate(MY_DATA))
# t2 = pipeline.load_dataset(MY_DATA)

## 11. Download results (CSV tables, PNG figures, interactive HTML)

In [ ]:
import shutil
archive = shutil.make_archive("pgmem_results", "zip", PROJECT)
print(archive)
try:
    from google.colab import files; files.download(archive)
except ImportError:
    pass